# 10단계 한눈에 보기

이 노트북은 작업하며 주고받은 답변(✍️ 칸과 클로드의 답변)을 빼고, 코드 · 결과 · 물음마다의 결론(📌)만 남긴 판이다. 단계마다 한 일과 정한 것은 README 「전체 흐름」에 있다.

### 말 뜻 — 표를 읽을 때 쓰는 말 (10-28에 통일함)

| 말 | 뜻 |
|---|---|
| 실제 | 그해(주로 2026년)에 실제로 발전한 양 · 실제 값입니다. 전력거래소 자료 그대로입니다 |
| 모델 예측 | 2023 ~ 2025년을 배운 모델이 그때의 날씨 · 시각 · 날 종류를 보고 예측한 양입니다. 9단계까지는 「평소라면」이라고 불렀습니다 |
| 실제 − 모델 예측 | 발전량 표의 숫자입니다. + 면 실제가 모델 예측보다 많았다, − 면 적었다는 뜻입니다 |
| 평소 해 | 2023 · 2024 · 2025년입니다. 「평소」는 이 세 해만 가리킵니다 |
| 평소 범위 | 평소 해에 모델 예측이 한 달 평균으로 가장 크게 틀린 크기(±2,177 MWh)입니다. 모델이 예측해 본 2024 · 2025년으로 냈습니다. 2026년의 차이가 이 안이면 「달랐다고 말할 근거가 없다」로 읽습니다 |
| 세 해 평균 | 2023 ~ 2025년 같은 달 값의 평균입니다. 모델 없이 계산한 것으로, 10단계 가격 표(2-가 · 2-라)에서만 씁니다 |

앞 단계 글과, 이 노트북에서 말을 통일하기 전에 쓴 글(표지 · 물음 1 · 2-나 답변 · 물음 2 결론 초안)에서 발전량을 두고 쓴 「평소라면」 · 「평소보다 많았다 · 적었다」 · 「실제 − 평소」는 「모델 예측」 · 「모델 예측보다 많았다 · 적었다」 · 「실제 − 모델 예측」으로 읽습니다.


# 10. 결과 해석 — 전체 11단계 중 10단계

**이 단계의 목적** — 9단계에서 낸 「2026년 1 ~ 6월 LNG는 평소 범위 안이었다」를 해석하는 단계입니다. 차이(MWh)를 LNG 톤과 배 몇 척으로 바꾸고, 그 옆에 같은 달 원전 · 석탄 발전량 변화를 놓고, 인과지도와 저량 · 유량 그림으로 이 프로젝트가 본 것과 보지 못한 것의 경계를 밝힙니다 — LNG가 들어오고 쌓이고 쓰이는 양 가운데 본 것은 LNG 발전량 하나입니다.

**이 노트북을 연 흐름** — 10-27에 9단계를 닫고 커밋한 뒤, 창주 님이 「lng 10단계 시작」이라고 하셨습니다. 물음 1의 선택지(무엇을 척으로 바꾸나 · 가정 셋을 어떻게 잡나)를 채팅으로 드리자, 창주 님이 「특정한 선택지가 어떤 결과 해석으로 이어질지 감이 잘 잡히지 않는데, 객관적으로 전체를 보고 있는 네가 봤을 때 가장 적합한게 뭐지. 교차검증하고 적합한지 최소 두 번 더 독립검토한다음에 너의 의견을 들려줘」라고 하셨고, 「독립적인 감사 후 의견을 제시받은 후에 같이 결정한 뒤에 노트북을 구성해야하지 않을까」라고 하셨습니다. 그래서 클로드가 검토 셋을 거쳐 의견을 먼저 드렸고(클로드 제안), 10-28에 창주 님이 「너가 가장 적합한 방향으로 가는 것으로 내가 판단한거거든?? 우리의 전체적인 목적에 걸맞게」라고 정하신 뒤 이 파일을 만들었습니다. 그 문답은 물음 1의 1-가에 옮겼습니다.

## 데이터와 모델

**데이터 흐름**

1. 2 · 3단계 — 시간마다 LNG · 기온 · 햇빛 같은 값을 받고, 빈 곳에 까닭을 붙여 고쳤습니다. 2026년 5월 2 ~ 17일은 값이 깨져 있어 비교에서 뺐습니다.
2. 4 · 5단계 — 무엇이 LNG를 움직이는지 보고, 모델이 쓸 열을 파일 하나(`data/hot/features_2023_2024_2025.csv`)로 저장했습니다.
3. 6단계 — 맞힐 값은 시간별 LNG 발전량(MWh) 그대로 두었습니다. 10단계에서 차이를 LNG 톤 · 배 몇 척으로 바꾸려면 MWh가 있어야 하기 때문입니다(6단계 5-가).
4. 7 · 8단계 — 2025년으로 채점하고 그래디언트 부스팅을 골랐습니다.
5. 9단계 — 그래디언트 부스팅을 2023 ~ 2025년으로 다시 배워 2026년 1 ~ 6월 시각마다 「평소라면 이만큼」을 맞히고 실제와 비교했습니다. 달마다 차이는 모두 평소 범위(평소 해에 가장 크게 틀린 달의 크기) 안이었고, 그 안에서 원전 발전량이 평소보다 적었던 것과 석탄 발전량이 평일에 평소보다 많았던 것이 상쇄되었습니다.
6. 10단계(이 노트북) — 9단계의 숫자를 준비 ② ~ ④에서 다시 내서 그 차이를 해석합니다. 9단계는 다음 단계가 읽을 표를 `data/hot`에 남기지 않아서, 같은 계산을 한 번 더 돌립니다.

**모델의 입력** (트리 계열 7열 — 6단계에서 정한 그대로)

- 몇 시인가 · 몇 월인가
- 무슨 날인가 — 요일 · 공휴일인가 · 명절인가
- 기온 · 햇빛 세기(일사) — 둘 다 서울 한 지점
- 넣지 않는 것 — 원전 · 석탄 · 총 전력거래량 · 가격 · 습도

**2026년 LNG를 쓰는 곳** — 준비 ③ · ④에서 9단계의 차이를 다시 내는 데만 씁니다. 모델은 고치지 않고, 달마다 한 시각 평균을 낸 뒤 시각마다의 값은 버립니다.

**먼저 준비 ①을 돌립니다.**

| 단계 | 이름 |
|---|---|
| 1 | 과제 정의 |
| 2 | 데이터 수집과 구성 보기 |
| 3 | 정제 |
| 4 | 탐색적 분석 |
| 5 | 변환·파생변수 |
| 6 | 분석계획 |
| 7 | 모델링과 검정 |
| 8 | 모델 확정과 핵심인자 |
| 9 | 모델 적용 |
| 10 | 결과 해석 ◀ 이 노트북 |
| 11 | 결론 |

진행 기록(지금 어디)은 리포의 `README.md`가 원본입니다. 단계마다 무엇을 하고 내가 무엇을 정하는지는 옵시디언 볼트 「빅데이터 분석의 순서」에 있습니다.

In [148]:
# 10단계 결과 해석 · 준비 ①
import os, glob                                # 폴더와 파일 경로를 다루는 기본 모듈 · 파일 이름을 한꺼번에 찾는 모듈
import numpy as np                             # 숫자 계산
import pandas as pd                            # 표(데이터프레임)를 다루는 라이브러리
from IPython.display import display            # 표를 글자가 아니라 격자로 그린다
pd.set_option("display.max_columns", 50)       # 열이 많아도 생략하지 않고 다 보이게

ROOT = os.getcwd()                             # 지금 폴더(보통 notebooks/)에서 시작해서
while not os.path.isdir(os.path.join(ROOT, "data")) and ROOT != os.path.dirname(ROOT):
    ROOT = os.path.dirname(ROOT)               # data 폴더가 보일 때까지 위로 = 프로젝트 루트
RAW = os.path.join(ROOT, "data", "raw")        # 원자료 폴더 — 2단계에서 받아 둔 것
HOT = os.path.join(ROOT, "data", "hot")        # 5단계가 만들어 둔 파일이 있는 폴더

print("프로젝트 루트:", ROOT)                    # 경로가 맞는지만 확인한다

프로젝트 루트: /Users/kimchangju/vs코드 /개인 프로젝트/lng-project


In [149]:
# 10단계 결과 해석 · 준비 ②
# ── 9단계 준비 ② · 1-다와 같은 계산 — 평소 해에 그래디언트 부스팅이 달마다 틀린 양과, 그 가운데 가장 큰 크기(평소 범위) ──
from sklearn.ensemble import HistGradientBoostingRegressor                   # 8단계에서 고른 그래디언트 부스팅
tbl = pd.read_csv(os.path.join(HOT, "features_2023_2024_2025.csv"), encoding="utf-8-sig")   # 5단계 「저장」 셀이 만든 표 — 6 ~ 9단계와 같은 파일
tbl["ts"] = pd.to_datetime(tbl["ts"])                                         # 글자로 저장된 시각을 다시 시각 자료형으로
tbl = tbl.set_index("ts").sort_index()                                        # 시각을 줄 이름으로 두고 시간 순서대로 놓는다
전체 = tbl.dropna(subset=["3시간 전 기온"])                                    # 3시간 전 기온이 빈 9시각은 뺀다 — 6 ~ 9단계와 같다
y = 전체["lng_mwh"]                                                           # 맞힐 값 — 그 시각의 LNG 발전량(MWh)
X = 전체[["hour24", "month", "icsr", "ta", "dow", "is_holiday", "명절"]]       # 트리 계열 7열 — 6단계에서 정하고 8단계에서 확정한 입력

def 그래디언트부스팅():                                                         # 8단계에서 확정한 설정 그대로
    return HistGradientBoostingRegressor(early_stopping=False, random_state=0)

평소 = {}                                                                     # 맞힌 해 → 시각마다 실제 − 예측(+ 면 실제가 예측보다 많았다)
for 맞힌해, 배운해 in [(2024, [2023]), (2025, [2023, 2024])]:                  # 2023 → 2024 · 2023 · 2024 → 2025 — 9단계와 같은 두 번
    배울곳 = 전체["year"].isin(배운해); 맞힐곳 = 전체["year"] == 맞힌해
    평소[맞힌해] = y[맞힐곳] - 그래디언트부스팅().fit(X[배울곳], y[배울곳]).predict(X[맞힐곳])
달평균 = pd.concat([잔차.groupby(잔차.index.month).mean() for 잔차 in 평소.values()])   # 스물세 달(2024년 8월은 자료가 없다)마다 시각마다 틀린 양의 평균
평소범위 = 달평균.abs().max()                                                    # + · −를 떼고 가장 큰 달 = 평소 범위(9단계 물음 1의 기준)
같다 = round(평소범위) == 2177                                                  # 9단계 1-다 [표 2]의 「그 달의 값」 크기
print(f"평소 범위를 다시 냈다 — 9단계 1-다의 2,177 MWh와 {'같다' if 같다 else '다르다(scikit-learn 판을 확인해 주세요)'}")

평소 범위를 다시 냈다 — 9단계 1-다의 2,177 MWh와 같다


In [150]:
# 10단계 결과 해석 · 준비 ③
# ── 9단계 준비 ③ · 2-가와 같은 계산 — 2026년 1 ~ 6월 「실제 − 모델 예측」을 달마다 한 시각 평균으로 ──
파일 = [os.path.join(RAW, "asos", "asos_108_202512.csv")] + sorted(glob.glob(os.path.join(RAW, "asos", "asos_108_2026*.csv")))   # 2025년 12월은 1월 1일 0 ~ 2시의 3시간 전 기온에만 쓴다
기상26 = pd.concat([pd.read_csv(f, encoding="utf-8-sig", usecols=["tm", "ta", "icsr"]) for f in 파일], ignore_index=True)   # 서울 ASOS — 시각 · 기온 · 일사만 읽는다
기상26["ts"] = pd.to_datetime(기상26["tm"])                                          # '2026-01-01 00:00' 글자를 시각으로
기상26 = 기상26.set_index("ts").sort_index()                                         # 시각을 줄 이름으로 두고 시간 순서대로
기상26["3시간 전 기온"] = 기상26["ta"].shift(freq="3h").reindex(기상26.index)           # 3시간 전 기온 — 5단계 물음 2와 같은 열
입력26 = 기상26.loc["2026-01-01 00:00":"2026-06-30 23:00"].drop(columns="tm").copy()   # 비교 기간 1 ~ 6월만 남긴다
입력26["icsr"] = 입력26["icsr"].fillna(0)                                              # 해가 없을 때 비는 일사는 0 — 3단계와 같다
입력26["year"] = 입력26.index.year                                                     # 2026
입력26["month"] = 입력26.index.month                                                   # 1 ~ 6월
입력26["hour24"] = 입력26.index.hour                                                   # 0 ~ 23시
입력26["dow"] = 입력26.index.dayofweek                                                 # 요일 0=월 … 6=일
휴일 = pd.read_csv(os.path.join(RAW, "holidays_kr.csv"), encoding="utf-8-sig")          # 2단계 `02_수집_공휴일`이 받아 둔 파일
날짜 = pd.Series(입력26.index.date, index=입력26.index)                                 # 시각마다 그 날짜
입력26["is_holiday"] = 날짜.isin(set(pd.to_datetime(휴일["date"]).dt.date)).astype(int)   # 공휴일이면 1 — 5단계와 같다
명절날짜 = set(pd.to_datetime(휴일.loc[휴일["name"].str.contains("설날|추석"), "date"]).dt.date)   # 이름에 설날 · 추석이 든 날
입력26["명절"] = 날짜.isin(명절날짜).astype(int)                                          # 설 · 추석 연휴면 1 — 5단계와 같다
요일 = 입력26["dow"]
입력26["날 종류"] = (요일.map({0: "월", 1: "화~금", 2: "화~금", 3: "화~금", 4: "화~금", 5: "토", 6: "일"})   # 5단계 「저장」 셀과 같은 여섯 가지
                    .where(~((입력26["is_holiday"] == 1) & (요일 <= 4) & (입력26["명절"] == 0)), "공휴일")    # 평일에 든 명절 아닌 공휴일
                    .where(입력26["명절"] == 0, "명절"))                                                  # 명절인 날은 요일과 상관없이
ta, ta3 = 입력26["ta"], 입력26["3시간 전 기온"]                                         # 지금 기온 · 3시간 전 기온
입력26["16도보다 추운 만큼"] = (16 - ta).clip(lower=0)                                  # 선형회귀(준비 ④)가 쓰는 열 — 5단계와 같다
입력26["24도보다 더운 만큼"] = (ta - 24).clip(lower=0)
입력26["3시간 전 16도보다 추운 만큼"] = (16 - ta3).clip(lower=0)
입력26["3시간 전 24도보다 더운 만큼"] = (ta3 - 24).clip(lower=0)

실제 = []                                                                              # 2026년 LNG 실제 값 — 연료 여덟이 24시간 다 온 날만(3단계와 같은 기준)
for f in sorted(glob.glob(os.path.join(RAW, "api", "2026*.csv"))):                     # 2026년 하루치 파일을 날짜 순서대로
    하루 = pd.to_datetime(os.path.basename(f)[:8])                                     # 파일 이름의 날짜(20260101)
    if 하루.month > 6 or os.path.getsize(f) < 100: continue                            # 비교 기간 1 ~ 6월만 · 0행으로 온 날은 건너뛴다
    t = pd.read_csv(f, encoding="utf-8-sig")                                           # 그날 파일을 읽는다
    if len(t) != 192: continue                                                         # 연료 여덟 × 24시간 = 192행이 아니면 쓰지 않는다 — 5월 2 ~ 17일이 여기 걸린다
    t = t[t["fuelTpCd"] == "LNG"]                                                      # LNG 줄만
    실제.append(pd.Series(t["amgo"].values, index=하루 + pd.to_timedelta(t["tradeNo"].astype(int) - 1, unit="h")))   # 1 ~ 24시를 0 ~ 23시 시각으로 — 3단계와 같다
실제 = pd.concat(실제).sort_index()                                                     # 시각마다 실제 LNG 발전량(MWh)
평소라면 = pd.Series(그래디언트부스팅().fit(X, y).predict(입력26.loc[실제.index, X.columns]), index=실제.index)   # 2023 ~ 2025년 세 해를 배운 모델 — 6단계 물음 5 · 9단계 2-가와 같다
차이 = 실제 - 평소라면                                                                   # + 면 실제가 모델 예측보다 많았다
달 = 차이.groupby(차이.index.month)                                                      # 달마다 묶는다
달차이 = pd.DataFrame({"비교한 날": 달.apply(lambda s: s.index.normalize().nunique()),     # 그 달에 실제 LNG가 다 온 날 수
                       "한 시각 평균": 달.mean()})                                        # 9단계 2-가 표의 「실제 − 모델 예측」과 같은 값
print("2026년 1 ~ 6월 「실제 − 모델 예측」을 다시 냈다 — 비교한 날: " + " · ".join(f"{m}월 {n}일" for m, n in 달차이["비교한 날"].items()))

2026년 1 ~ 6월 「실제 − 모델 예측」을 다시 냈다 — 비교한 날: 1월 31일 · 2월 28일 · 3월 31일 · 4월 30일 · 5월 15일 · 6월 30일


In [151]:
# 10단계 결과 해석 · 준비 ④
# ── 9단계 1-사 · 2-나와 같은 계산 — 선형회귀로 「실제 − 평소라면」을 수요 · 원전 · 석탄 · 그 밖 넷으로 나눈다 ──
from sklearn.linear_model import LinearRegression                                    # 선형회귀 — 7단계와 같다
def 선형열(d):                                                                        # 7단계 준비 ②의 「선형열」과 같은 67열
    시각열 = pd.get_dummies(d["hour24"].astype(str), prefix="시", drop_first=True).astype(float)   # 1~23시 0/1 열 — 0시가 기준
    월열 = pd.get_dummies(d["month"].astype(str), prefix="월", drop_first=True).astype(float)      # 2~12월 0/1 열 — 1월이 기준
    날열 = pd.get_dummies(d["날 종류"], prefix="날").astype(float).drop(columns="날_월")            # 화~금·토·일·공휴일·명절 — 월요일이 기준
    쉬는날 = d["날 종류"].isin(["토", "일", "공휴일", "명절"]).astype(float)                         # 쉬는 날이면 1
    쉬는날시간대 = (시각열 - 1 / 24).mul(쉬는날, axis=0)                                            # 5단계 물음 7 — 쉬는 날 × 시간대
    쉬는날시간대.columns = [f"쉬는 날×{c[2:]}시" for c in 시각열.columns]
    기상 = pd.DataFrame({"16도보다 추운 만큼": d["16도보다 추운 만큼"], "24도보다 더운 만큼": d["24도보다 더운 만큼"],
                         "3시간 동안 바뀐 만큼 (추운 쪽)": d["16도보다 추운 만큼"] - d["3시간 전 16도보다 추운 만큼"],
                         "3시간 동안 바뀐 만큼 (더운 쪽)": d["24도보다 더운 만큼"] - d["3시간 전 24도보다 더운 만큼"],
                         "일사": d["icsr"]})
    return pd.concat([시각열, 월열, 날열, 기상, 쉬는날시간대], axis=1)
선형X = 선형열(전체)                                                                  # 준비 ②의 「전체」로 만든다

석탄종류 = ["유연탄", "무연탄"]                                                          # 석탄 = 유연탄 + 무연탄
def 연간(해):                                                                            # 전력거래소 연간 파일 한 해 — 시각마다 「LNG 밖 모든 발전원의 합」과 「석탄」
    이름 = [f for f in os.listdir(os.path.join(RAW, "kpx")) if f"{해}1231" in f][0]       # 파일 이름에 20251231 같은 기준일이 있다
    d = pd.read_csv(os.path.join(RAW, "kpx", 이름), encoding="utf-8-sig")
    d.columns = ["date", "hour", "fuel", "cap_mw", "mwh"]                               # 해마다 열 이름이 달라서 순서로 통일한다(3단계와 같다)
    d["fuel"] = d["fuel"].str.strip()
    d = d[d["fuel"] != "LNG"]                                                            # LNG 줄은 읽자마자 버린다 — LNG는 준비 ②의 값을 쓴다
    ts = pd.to_datetime(d["date"]) + pd.to_timedelta(d["hour"].astype(int) - 1, unit="h")   # 1 ~ 24시를 0 ~ 23시 시각으로
    석 = d["fuel"].isin(석탄종류)
    return d["mwh"].groupby(ts).sum(), d["mwh"][석].groupby(ts[석]).sum()
def API(해, 끝달):                                                                       # API 하루치 파일 — 연료 여덟이 24시간 다 온 날만, 발전원마다 한 열
    조각 = []
    for f in sorted(glob.glob(os.path.join(RAW, "api", f"{해}*.csv"))):
        하루 = pd.to_datetime(os.path.basename(f)[:8])
        if 하루.month > 끝달 or os.path.getsize(f) < 100: continue                       # 그 달까지만 · 0행으로 온 날은 건너뛴다
        t = pd.read_csv(f, encoding="utf-8-sig")
        if len(t) != 192: continue                                                       # 연료 여덟 × 24시간이 아닌 날은 쓰지 않는다
        t["ts"] = 하루 + pd.to_timedelta(t["tradeNo"].astype(int) - 1, unit="h")
        조각.append(t.pivot_table(index="ts", columns="fuelTpCd", values="amgo", aggfunc="sum"))
    return pd.concat(조각).sort_index().drop(columns="LNG")                              # LNG 열은 버린다 — LNG는 준비 ② · ③의 값을 쓴다
a24, a26 = API(2024, 12), API(2026, 6)
밖23, 석23 = 연간(2023)
밖25, 석25 = 연간(2025)
LNG밖 = pd.concat([밖23, a24.sum(axis=1), 밖25]).sort_index().reindex(전체.index)        # 2023 ~ 2025년 시각마다 LNG를 뺀 모든 발전원의 합
석탄 = pd.concat([석23, a24[석탄종류].sum(axis=1), 석25]).sort_index().reindex(전체.index)
원전 = 전체["nuke_mwh"]                                                                  # 7단계 3-거와 같은 원전
대상 = {"수요": y + LNG밖, "원전": 원전, "석탄": 석탄, "그 밖": LNG밖 - 원전 - 석탄}         # 수요 = LNG + LNG 밖 모든 발전원 · 그 밖 = 수력 · 양수 · 유류 · 태양광 · 풍력 같은 것
a26 = a26.reindex(실제.index)                                                            # 준비 ③과 같은 시각
대상26 = {"수요": 실제 + a26.sum(axis=1), "원전": a26["원자력"], "석탄": a26[석탄종류].sum(axis=1),
          "그 밖": a26.sum(axis=1) - a26["원자력"] - a26[석탄종류].sum(axis=1)}
assert all(v.notna().all() for v in list(대상.values()) + list(대상26.values())), "빈 시각이 있다"

X선26 = 선형열(입력26).reindex(columns=선형X.columns, fill_value=0).loc[실제.index]       # 2026년에는 7 ~ 12월 열이 없어서 0으로 채운다
배운곳 = 전체["year"].isin([2023, 2024, 2025])                                           # 그래디언트 부스팅과 같은 세 해를 배운다
평소라면값 = lambda 값: LinearRegression().fit(선형X[배운곳], 값[배운곳]).predict(X선26)   # 같은 67열로 그 값의 「평소라면」을 맞힌다
몫 = pd.DataFrame({"수요": 대상26["수요"] - 평소라면값(대상["수요"])}, index=실제.index)    # 수요가 평소보다 많으면 + — LNG가 늘었다
for k in ["원전", "석탄", "그 밖"]:
    몫[k] = -(대상26[k] - 평소라면값(대상[k]))                                           # 발전원이 평소보다 적으면 + — 그만큼 LNG가 늘었다
몫["선형회귀"] = 몫[["수요", "원전", "석탄", "그 밖"]].sum(axis=1)                        # 넷을 더하면 선형회귀가 본 LNG 차이
어긋남 = (몫["선형회귀"] - (실제 - 평소라면값(y))).abs().max()                             # 선형회귀가 LNG를 바로 맞힌 차이와 넷의 합 — 0이어야 한다
달몫 = 몫.groupby(몫.index.month).mean()                                                 # 달마다 한 시각 평균 — 9단계 2-나 [표 1]과 같은 값
del 실제, 평소라면, 차이, 몫                                                             # 2026년 시각마다의 LNG 값은 여기서 버린다 — 10단계에는 달마다 한 시각 평균만 남긴다
print(f"선형회귀로 넷으로 나눴다 — 넷의 합과 선형회귀 LNG 차이가 가장 크게 어긋난 시각도 {어긋남:.6f} MWh (0이어야 한다)")

선형회귀로 넷으로 나눴다 — 넷의 합과 선형회귀 LNG 차이가 가장 크게 어긋난 시각도 0.000000 MWh (0이어야 한다)


# 물음 1 — 9단계의 차이를 LNG 몇 톤 · 몇 척으로 바꾸면 얼마인가

<div style="border: 2px solid var(--vscode-focusBorder, #3794ff); border-left-width: 6px; border-radius: 8px; padding: 4px 18px 10px; background: var(--vscode-textBlockQuote-background, rgba(127, 127, 127, 0.1));">

## 📌 물음 1 결론

**왜 생긴 물음인가** — 9단계의 차이는 「한 시각 평균 몇 MWh」라서, LNG를 사는 쪽에서는 그 크기가 큰지 작은지 읽히지 않습니다. 지침 10단계 칸과 1단계 물음 2에서 그 차이를 LNG 톤과 배 수로 바꾸기로 했고, 그 크기를 정하는 물음입니다.

### 결정 — 평소 범위는 배 약 3 ~ 4척(LNG 약 23 ~ 25만 톤, 30일 기준)이고, 2026년 1 ~ 6월은 가장 크게 달랐던 6월도 약 2 ~ 3척(약 17 ~ 18만 톤) 줄어든 정도라 달마다 그 안이었다

**같이 정한 것**

1. 배로 바꾸는 것은 9단계의 차이와 평소 범위 둘이고, 둘 다 30일 기준으로 같은 값을 곱한다. 선형회귀로 나눈 네 줄은 GWh로 둔다(1-가 — 맡기셔서 클로드가 정함).
2. 가정은 값 하나가 아니라 범위로 둔다. 효율은 한국전력통계 값 하나(45.46%)이고, 범위는 그 통계가 어느 발열량 기준인지 모르는 만큼과 배 크기에서 생긴다 — 배 한 척 약 356 ~ 497 GWh(1-나). 처음 둔 프랑스 발전소의 기록 효율은 한국 자료에 맞지 않아 1-나를 본 뒤에 뺐다(1-나 「✍️ 다시 읽고 나서 (2)」).
3. 새로 나오는 판정은 없다 — 차이와 범위에 같은 값을 곱해서 9단계 판정이 그대로다. 물음 1이 더한 것은 크기를 말하는 단위다(1-다 답변).

**결정 근거**

- 1-다 [표 1] 「평소 범위 안인가」 — 1 ~ 6월 모두 안.
- 1-다 [표 1] 「평소 범위」 줄 — ±3.2 ~ ±4.4척 · ±22.7 ~ ±25.1만 톤. 6월 줄 — −2.3 ~ −3.2척 · −16.6 ~ −18.4만 톤.
- 창주 님 — 1-다 「✍️ 내 답변」 「특이점이 있다고 말할 수 있는게 없다는 것」 · 「✍️ 정할 것」 「그래 위 해석방향으로 가자」.

**남은 것**

- 왜 평소 범위 안이었나(원전이 적었던 것과 석탄이 많았던 것이 상쇄된 까닭)는 물음 2에서 봤다(📌 물음 2 결론).
- 실제 배 척 수는 범위의 많은 쪽에 가깝거나 그보다 많을 수 있다(판단) — 11단계 한계(닫는 칸 ②).

</div>


## 1-가 — 무엇을 척으로 바꾸고 가정 셋을 어떻게 잡을지 정한 문답 (10-27 ~ 28 채팅)

**이 셀이 답하는 물음** — 9단계의 차이를 척으로 바꿀 때 무엇을 바꾸나(첫째), 가정 셋(발열량 · 효율 · 배 한 척 물량)을 어떻게 잡나(둘째).

**첫째 — 무엇을 척으로 바꾸나** (n · a ~ b · c · d는 지어낸 표시이고 실제 숫자가 아닙니다)

| | 가 · 차이만 | 나 · 차이와 평소 범위 (네 줄은 GWh) | 다 · 차이와 평소 범위 (네 줄도 척) |
|---|---|---|---|
| 결론 문장 | 「2026년 ○월은 평소보다 n척 더 썼다」 | 「평소 해에도 가장 크게 틀린 달은 a ~ b척(30일 기준)만큼 어긋났고, 2026년 1 ~ 6월은 달마다 그 안이었다. 그 안에서 원전 발전량이 평소보다 적어 LNG 발전량이 늘고, 석탄 발전량이 평소보다 많아 줄어 상쇄되었다」 | 나와 같고, 뒤 문장만 「… LNG가 c척 늘고 … d척 줄어 상쇄되었다」로 바뀐다 |
| 선형회귀로 나눈 네 줄 (수요 · 원전 · 석탄 · 그 밖) | 쓰지 않는다 | 30일 기준 GWh로 옆에 둔다 | 척으로 바꾼다 |
| 대가 | 평소에도 그만큼은 어긋난다는 것이 안 보여서, 평소 범위 안인 차이가 「더 썼다」로 읽힌다 | 네 줄은 척으로 바로 읽히지 않는다 | 네 줄은 선형회귀로 나눈 몫이라 더해도 그래디언트 부스팅의 차이와 같지 않다. 평소 해에도 그만큼 나오던 줄을 「원전 때문에 c척」으로 쓰면 실제보다 크게 읽힌다 |

**둘째 — 가정 셋을 어떻게 잡나**

| | 가 · 공식 값 하나씩 | 나 · 낮은 값과 높은 값 | 다 · 자료로 직접 「1 MWh에 몇 톤」 |
|---|---|---|---|
| 결론 문장 | 「n척」 | 「a ~ b척」 | 「n척」 |
| 대가 | 효율을 무엇 기준으로 셌는지 확인되지 않아, 값 하나가 실제보다 정밀해 보인다 | 결론 문장이 범위가 된다 | 새 자료(가스공사 월별 공급량)를 받아야 하고, 그 자료는 직수입이 빠지면 작게, 열병합 발전소의 난방열 연료가 섞이면 크게 나와 얼마나 틀리는지 알 수 없다 |

## 1-나 — 가정 셋(발열량 · 효율 · 배 한 척 물량)의 값과 출처

**이 셀이 답하는 물음** — MWh를 LNG 톤과 배 몇 척으로 바꿀 때 쓰는 값은 무엇이고, 어디서 왔고, 얼마나 확인됐나.

**고르는 규칙** (값을 찾기 전에 정한 것 — 넷째 · 다섯째 줄은 10-28에 이 셀을 본 뒤에 바꿨습니다)

- 찾는 것은 셋입니다 — 발열량(LNG 1 kg에서 나오는 열), 효율(그 열 가운데 전기로 바뀌는 비율), 배 한 척 물량(톤).
- 셋마다 「척이 적게 나오는 쪽」과 「척이 많이 나오는 쪽」을 같이 찾습니다.
- 효율은 우리 자료와 같은 송전단(발전소 밖으로 내보낸 전기) 기준으로 맞춥니다. 우리 자료(전력거래량)가 송전단입니다.
- 효율은 한국 발전소의 값(한국전력통계)만 씁니다. 처음에는 새 발전소의 세계 기록(프랑스, 62.22%)을 한쪽 끝으로 두었는데, 한국 발전소를 나타내는 값이 아니고 평소와 달랐던 만큼에 맞는 값도 아니어서 뺐습니다(1-나 「✍️ 다시 읽고 나서 (2)」).
- 효율과 발열량은 같은 기준(총발열량끼리 · 순발열량끼리)으로 짝짓습니다. 한국전력통계 효율은 어느 기준인지 적혀 있지 않아서, 두 기준을 범위의 두 끝으로 둡니다.
- 한국전력통계 효율은 2023 ~ 2025년 세 해 평균을 씁니다 — 모델이 배운 해와 같습니다.

**표의 숫자**

- [표 1] 가정마다 값 · 출처 · 확인 정도입니다. 「확인」은 원본을 직접 본 것, 「판단」은 원본에 적혀 있지 않아 클로드가 판단한 것, 「확인 불가」는 찾지 못한 것입니다.
- [표 2] 두 끝입니다. 「1 GWh에 드는 LNG」는 1,000 MWh × 3.6 GJ/MWh ÷ 효율 ÷ 발열량(GJ/톤)이고, 「1척이 몇 GWh」는 배 한 척 ÷ 1 GWh에 드는 LNG입니다. 효율은 두 끝이 같고, 발열량이 클수록(총발열량) · 배가 클수록 척이 적게 나옵니다.
- 모델끼리 비교하는 표가 아닙니다. 9단계의 차이를 척으로 바꿀 때 곱할 값을 정하는 표입니다.

In [152]:
# 10단계 결과 해석 · 1-나
# ── 가정 셋 — 값 · 출처 · 확인 정도(확인 = 원본을 직접 봄) ──
발열량 = {"총발열량": 54.7, "순발열량": 49.4}                                       # LNG 1 kg에서 나오는 열(MJ) — 1톤당 GJ와 같은 숫자다 · 에너지법 시행규칙 별표(2022.11.21 개정)
통계 = {2023: 45.38218890, 2024: 45.29090762, 2025: 45.69903185}                    # 한전 자회사 복합화력의 송전단 열효율(%) — EPSIS 「화력발전소 열효율」, 출처 한국전력통계 책자
효율 = np.mean(list(통계.values())) / 100                                            # 2023 ~ 2025년 세 해 평균 — 효율은 한국 발전소의 값 하나만 쓴다
한척 = {"125,000 m³": 57_000, "135,000 m³": 62_000, "170,000 m³": 72_000}           # 배 한 척에 싣는 LNG(톤)

표1 = pd.DataFrame([                                                                 # [표 1] 가정마다 값 · 출처 · 확인 정도
    ["발열량 · 총발열량 (물이 증발할 때의 열까지 넣은 값)", "54.7 MJ/kg", "에너지법 시행규칙 별표 (2022.11.21 개정)", "확인"],
    ["발열량 · 순발열량 (그 열을 뺀 값)", "49.4 MJ/kg", "같은 표", "확인"],
    ["효율 · 한국전력통계 2023 ~ 2025년 평균", f"{효율 * 100:.2f}%", "EPSIS 「화력발전소 열효율」 복합화력 송전단 열 (한전 자회사로 보임)", "값과 송전단은 확인 · 한전 자회사인지는 판단 · 총/순 기준은 확인 불가"],
    ["배 한 척 · 125,000 m³", "57,000톤", "한국가스공사 누리집", "확인"],
    ["배 한 척 · 135,000 m³", "62,000톤", "한국가스공사 누리집", "확인"],
    ["배 한 척 · 170,000 m³ 만재", "72,000톤", "Riviera", "확인"],
], columns=["가정", "값", "출처", "확인 정도"]).set_index("가정")

def 톤GWh(효, 발):                                                                    # 1 GWh를 만드는 데 드는 LNG(톤)
    return 1000 * 3.6 / 효 / 발                                                      # 1,000 MWh × 3.6 GJ/MWh ÷ 효율 ÷ 발열량(GJ/톤)
끝 = {"척이 적게 나오는 끝": ("총발열량", 한척["170,000 m³"]),                         # 통계 효율이 총발열량 기준이라면 · 배가 가장 클 때
      "척이 많이 나오는 끝": ("순발열량", 한척["125,000 m³"])}                         # 통계 효율이 순발열량 기준이라면 · 배가 가장 작을 때
줄들 = []                                                                              # [표 2]의 줄을 모을 목록
for k, (발, 척) in 끝.items():
    t = 톤GWh(효율, 발열량[발])                                                        # 1 GWh에 드는 LNG(톤)
    줄들.append({"끝": k, "효율": f"{효율 * 100:.2f}%", "발열량": f"{발} {발열량[발]} MJ/kg",
                 "1 GWh에 드는 LNG (톤)": f"{t:,.1f}", "배 한 척 (톤)": f"{척:,}", "1척이 몇 GWh": f"{척 / t:,.1f}"})
두끝 = pd.DataFrame(줄들).set_index("끝")
톤값 = {k: 톤GWh(효율, 발열량[발]) for k, (발, 척) in 끝.items()}                       # 1 GWh에 드는 LNG(톤) — 척으로 바꿀 때 쓴다
척값 = {k: 척 / 톤값[k] for k, (발, 척) in 끝.items()}                                  # 1척이 몇 GWh — 척으로 바꿀 때 쓴다

print("[표 1] 가정 셋 — 확인 = 원본을 직접 봄")
display(표1)
print("[표 2] 두 끝 — 1 GWh에 드는 LNG = 1,000 MWh × 3.6 GJ/MWh ÷ 효율 ÷ 발열량, 1척이 몇 GWh = 배 한 척 ÷ 1 GWh에 드는 LNG")
print("       평소와 달랐던 만큼의 전기는 평균보다 효율이 낮은 발전소에서 나와서, 실제 배 척 수는 「척이 많이 나오는 끝」보다 많을 수 있다(판단)")
display(두끝)

[표 1] 가정 셋 — 확인 = 원본을 직접 봄


,값,출처,확인 정도
가정,,,
발열량 · 총발열량 (물이 증발할 때의 열까지 넣은 값),54.7 MJ/kg,에너지법 시행규칙 별표 (2022.11.21 개정),확인
발열량 · 순발열량 (그 열을 뺀 값),49.4 MJ/kg,같은 표,확인
효율 · 한국전력통계 2023 ~ 2025년 평균,45.46%,EPSIS 「화력발전소 열효율」 복합화력 송전단 열 (한전 자회사로 보임),값과 송전단은 확인 · 한전 자회사인지는 판단 · 총/순 기준은 확인 불가
"배 한 척 · 125,000 m³","57,000톤",한국가스공사 누리집,확인
"배 한 척 · 135,000 m³","62,000톤",한국가스공사 누리집,확인
"배 한 척 · 170,000 m³ 만재","72,000톤",Riviera,확인


[표 2] 두 끝 — 1 GWh에 드는 LNG = 1,000 MWh × 3.6 GJ/MWh ÷ 효율 ÷ 발열량, 1척이 몇 GWh = 배 한 척 ÷ 1 GWh에 드는 LNG
       평소와 달랐던 만큼의 전기는 평균보다 효율이 낮은 발전소에서 나와서, 실제 배 척 수는 「척이 많이 나오는 끝」보다 많을 수 있다(판단)


,효율,발열량,1 GWh에 드는 LNG (톤),배 한 척 (톤),1척이 몇 GWh
끝,,,,,
척이 적게 나오는 끝,45.46%,총발열량 54.7 MJ/kg,144.8,"72,000",497.3
척이 많이 나오는 끝,45.46%,순발열량 49.4 MJ/kg,160.3,"57,000",355.6


## 1-다 — 9단계의 차이와 평소 범위를 배 몇 척으로

**이 셀이 답하는 물음** — 9단계의 달마다 차이와 평소 범위는 LNG 배 몇 척인가.

**표의 숫자**

- [표 1] 달마다 9단계의 차이 「실제 − 모델 예측」(한 시각 평균 MWh — + 면 실제가 모델 예측보다 많았다)을 30일 기준 GWh로 바꾸고, 1-나의 두 끝으로 LNG 만 톤과 척을 냅니다. 만 톤은 GWh × 「1 GWh에 드는 LNG」(1-나 [표 2]) ÷ 10,000, 척은 GWh ÷ 「1척이 몇 GWh」입니다. 맨 아래 줄은 평소 범위(평소 해에 그래디언트 부스팅이 가장 크게 틀린 달의 크기)를 같은 방법으로 바꾼 것입니다.
- 30일 기준은 한 시각 평균에 720시각을 곱한 양입니다. 그 달의 실제 양이 아니고, 달마다 날 수가 달라도(2월 28일 · 5월 15일) 평소 범위와 같은 기준으로 비교하려고 이렇게 바꿉니다. 5월은 15일 평균을 30일로 늘린 값입니다.
- 톤과 척은 발전에 쓴 LNG를 바꾼 양이고, 들여온 LNG의 양이나 배의 수가 아닙니다. 들여오는 양은 계약 물량 · 저장 재고 · 도시가스 수요로도 정해집니다.
- 「평소 범위 안인가」는 9단계와 같은 판정입니다. 차이와 범위에 같은 수를 곱해서, 척으로 바꿔도 바뀌지 않습니다.
- [표 2] 9단계 2-나의 나눈 표를 같은 30일 기준 GWh로 바꾼 것입니다. 맨 위 줄은 [표 1]의 차이를 GWh로 적은 것이고, 둘째 줄은 선형회귀가 본 차이(아래 네 줄의 합)입니다. 네 줄은 선형회귀로 나눈 몫이라 척으로 바꾸지 않습니다.
- 모델끼리 비교하는 표가 아닙니다.

In [153]:
# 10단계 결과 해석 · 1-다
# ── 9단계의 차이와 평소 범위를 척으로 — 둘 다 30일 기준(한 시각 평균 × 720시각)으로, 1-나의 같은 값을 곱한다 ──
def GWh30(mwh): return mwh * 720 / 1000                                              # 한 시각 평균(MWh) × 720시각 ÷ 1,000 = 30일 기준(GWh)
척적게, 척많이 = 1 / 척값["척이 적게 나오는 끝"], 1 / 척값["척이 많이 나오는 끝"]        # 1 GWh당 척 — 1-나 [표 2]의 「1척이 몇 GWh」를 뒤집은 값
줄 = []
for m, r in 달차이.iterrows():                                                        # 달마다
    g = GWh30(r["한 시각 평균"])
    줄.append({"달": f"{m}월", "비교한 날": f"{int(r['비교한 날'])}일",
               "실제 − 모델 예측 · 한 시각 평균 (MWh)": f"{r['한 시각 평균']:+,.0f}",
               "30일 기준 (GWh)": f"{g:+,.0f}",
               "만 톤 · 적게 나오는 끝": f"{g * 톤값['척이 적게 나오는 끝'] / 1e4:+.1f}",      # GWh × 1 GWh에 드는 LNG(톤) ÷ 10,000
               "만 톤 · 많이 나오는 끝": f"{g * 톤값['척이 많이 나오는 끝'] / 1e4:+.1f}",
               "척 · 적게 나오는 끝": f"{g * 척적게:+.1f}", "척 · 많이 나오는 끝": f"{g * 척많이:+.1f}",
               "평소 범위 안인가": "안" if abs(r["한 시각 평균"]) <= 평소범위 else "밖"})   # 9단계와 같은 판정 — 같은 수를 곱해도 바뀌지 않는다
g = GWh30(평소범위)
줄.append({"달": "평소 범위", "비교한 날": "—", "실제 − 모델 예측 · 한 시각 평균 (MWh)": f"±{평소범위:,.0f}",
           "30일 기준 (GWh)": f"±{g:,.0f}",
           "만 톤 · 적게 나오는 끝": f"±{g * 톤값['척이 적게 나오는 끝'] / 1e4:.1f}", "만 톤 · 많이 나오는 끝": f"±{g * 톤값['척이 많이 나오는 끝'] / 1e4:.1f}",
           "척 · 적게 나오는 끝": f"±{g * 척적게:.1f}", "척 · 많이 나오는 끝": f"±{g * 척많이:.1f}", "평소 범위 안인가": "—"})
표1 = pd.DataFrame(줄).set_index("달")

이름 = {"그래디언트 부스팅": "그래디언트 부스팅 — LNG가 모델 예측과 달랐던 양 (위 표와 같은 값)",
        "선형회귀": "선형회귀 — 아래 넷을 더한 값",
        "수요": "수요 줄 — + 면 수요가 모델 예측보다 많아서 LNG가 늘었다",
        "원전": "원전 줄 — + 면 원전 발전량이 모델 예측보다 적어서 LNG가 늘었다",
        "석탄": "석탄 줄 — + 면 석탄 발전량이 모델 예측보다 적어서 LNG가 늘었다",
        "그 밖": "그 밖 줄 — + 면 그 밖 발전량이 모델 예측보다 적어서 LNG가 늘었다"}
표2 = 달몫.assign(**{"그래디언트 부스팅": 달차이["한 시각 평균"]})[list(이름)].T.rename(index=이름)   # 9단계 2-나 [표 1]과 같은 줄
표2 = 표2.apply(lambda c: GWh30(c).map("{:+,.0f}".format))                                  # 30일 기준 GWh로
표2.columns = [f"{m}월" for m in 표2.columns]

print("[표 1] 2026년 달마다 LNG가 모델 예측과 달랐던 양을 톤과 척으로 — 30일 기준은 그 달의 실제 양이 아니라 한 시각 평균 × 720시각")
print("       5월은 15일 평균을 30일로 늘린 값 · 톤과 척은 발전에 쓴 LNG를 바꾼 양이고, 들여온 LNG의 양이나 배의 수가 아니다")
display(표1)
print("[표 2] 같은 30일 기준 GWh로 본 9단계 2-나의 나눈 표 — 네 줄은 선형회귀로 나눈 몫이라 척으로 바꾸지 않는다")
print(f"       척으로 읽고 싶으면 1척 ≈ {1 / 척많이:,.0f} ~ {1 / 척적게:,.0f} GWh (1-나의 두 끝)")
display(표2)

[표 1] 2026년 달마다 LNG가 모델 예측과 달랐던 양을 톤과 척으로 — 30일 기준은 그 달의 실제 양이 아니라 한 시각 평균 × 720시각
       5월은 15일 평균을 30일로 늘린 값 · 톤과 척은 발전에 쓴 LNG를 바꾼 양이고, 들여온 LNG의 양이나 배의 수가 아니다


,비교한 날,실제 − 모델 예측 · 한 시각 평균 (MWh),30일 기준 (GWh),만 톤 · 적게 나오는 끝,만 톤 · 많이 나오는 끝,척 · 적게 나오는 끝,척 · 많이 나오는 끝,평소 범위 안인가
달,,,,,,,,
1월,31일,"+1,096",+789,+11.4,+12.7,+1.6,+2.2,안
2월,28일,"+1,446","+1,041",+15.1,+16.7,+2.1,+2.9,안
3월,31일,"+1,211",+872,+12.6,+14.0,+1.8,+2.5,안
4월,30일,+257,+185,+2.7,+3.0,+0.4,+0.5,안
5월,15일,+38,+27,+0.4,+0.4,+0.1,+0.1,안
6월,30일,"-1,595","-1,148",-16.6,-18.4,-2.3,-3.2,안
평소 범위,—,"±2,177","±1,568",±22.7,±25.1,±3.2,±4.4,—


[표 2] 같은 30일 기준 GWh로 본 9단계 2-나의 나눈 표 — 네 줄은 선형회귀로 나눈 몫이라 척으로 바꾸지 않는다
       척으로 읽고 싶으면 1척 ≈ 356 ~ 497 GWh (1-나의 두 끝)


,1월,2월,3월,4월,5월,6월
그래디언트 부스팅 — LNG가 모델 예측과 달랐던 양 (위 표와 같은 값),+789,"+1,041",+872,+185,+27,"-1,148"
선형회귀 — 아래 넷을 더한 값,+809,"+1,008",+746,+235,+194,"-1,235"
수요 줄 — + 면 수요가 모델 예측보다 많아서 LNG가 늘었다,-419,-774,"-1,214",-582,+6,"-1,154"
원전 줄 — + 면 원전 발전량이 모델 예측보다 적어서 LNG가 늘었다,"+1,551","+2,625","+2,749","+1,059","+1,618",+500
석탄 줄 — + 면 석탄 발전량이 모델 예측보다 적어서 LNG가 늘었다,-5,-561,-804,-238,"-2,039","-1,072"
그 밖 줄 — + 면 그 밖 발전량이 모델 예측보다 적어서 LNG가 늘었다,-319,-282,+15,-4,+608,+491


# 물음 2 — 석탄이 2026년에 모델 예측보다 많았던 까닭은 무엇인가

<div style="border: 2px solid var(--vscode-focusBorder, #3794ff); border-left-width: 6px; border-radius: 8px; padding: 4px 18px 10px; background: var(--vscode-textBlockQuote-background, rgba(127, 127, 127, 0.1));">

## 📌 물음 2 결론

**왜 생긴 물음인가** — 9단계에서 2026년 1 ~ 6월 LNG 발전량은 평소 범위 안이었고, 그 안에서 원전 발전량이 모델 예측보다 적었던 것과 석탄 발전량이 모델 예측보다 많았던 것이 상쇄됐습니다. 원전 쪽은 멈춘 원전이 까닭 후보였지만, 석탄이 많았던 까닭은 찾지 못했습니다. 그 까닭을 찾는 물음입니다.

### 결정 — 석탄 발전량이 모델 예측보다 많았던 것은 값 차이로 설명되지 않는다. 석탄과 LNG가 더 만든 양 가운데 석탄의 몫은 1월 1%에서 5월 91%로 늘었고, 시기가 맞는 까닭 셋은 데이터로 가를 수 없다

**창주 님이 정한 결론 문장** — 「석탄 발전량이 2026년에 모델 예측보다 많았던 것은 LNG와 석탄의 값 차이로는 설명되지 않는다(1 ~ 4월에는 값 차이가 평소 해보다 좁았고, 5 · 6월에는 평소 해와 비슷했다). 원전과 그 밖 발전원을 더한 발전량은 1 ~ 6월 모두 모델 예측보다 적었고, 그 양은 석탄이 더 만든 양, LNG가 더 만든 양, 전력 수요가 줄어 채울 필요가 없었던 양으로 나뉜다. 석탄과 LNG가 더 만든 양 가운데 석탄의 몫은 1월 1%, 2월 36%, 3월 52%, 4월 50%, 5월 91%로 1월부터 늘었다. 6월에는 수요가 줄어든 양이 원전과 그 밖이 덜 만든 양보다 컸고, 석탄은 더 만들고 LNG는 덜 만들었다(석탄이 LNG 대신 만든 것으로 본다 — 판단). 석탄의 몫이 늘어난 것과 시기가 맞는 것은 셋이다 — 겨울 계절관리제와 3월 16일 석탄 상한 해제, 1 ~ 4월에 좁았던 값 차이가 5월에 평소 해와 같아진 것, 3월부터의 정부의 LNG 수요 관리. 셋을 데이터로 가를 수는 없다(판단). 「전력 수요가 줄었다」는 전력시장에서 거래된 양이 줄었다는 뜻이고, 전기를 덜 썼는지 전력시장 밖의 태양광이 채웠는지는 가를 수 없다.」 (클로드가 제안하고 창주 님이 「좋아」로 정함 → 2-바 · 2-사를 본 뒤 「논리나 설명에 따르면 바꾼게 더 적합해보여」로 고침 → 2-아를 본 뒤 「2번이 더 적합한거 아닌가??」 · 「해석이니까 디테일하게 적는게 맞겠다. 차라리 비유적인 표현 지우고 좀 딱딱하더라도 적는게 맞겠어」로 다시 고침)

**같이 정한 것**

1. 1단계 짐작 ⑥ 「결국 상대 가격이 결정한다」 · ④ 「연료별 효율과 가격 비중, 경제성」 — 「어느 쪽이 싼가」로는 2026년 1 ~ 6월에 맞지 않았다. 석탄이 1 kWh당 연료비단가로 늘 LNG보다 쌌는데도 석탄의 몫이 1월 1%에서 5월 91%로 달마다 달랐다(창주 님 「그러면 이게 답변이 일정하지 않으니까 가정대로 현실화되지 않았다고 말할수있는거 아녀?」). 다만 값 차이의 크기(1 ~ 4월에 좁았다가 5월에 평소 해와 같아졌다)는 석탄의 몫이 늘어난 것과 시기가 맞아서, 까닭 셋 가운데 하나로 남는다(판단).
2. 1단계 짐작 ⑤ — 창주 님이 1단계 5번에 적으신 물음(「…lng가 줄었다면 다른 곳에서 에너지원을 가져와야 했을 것인데 어디서 들고왔을까?」)은 이 프로젝트로는 알 수 없다. 자료에는 발전소가 쓴 LNG만 있고 들여온 LNG는 없다(창주 님 「그러면 이거 데이터 없으면 아니 이 프로젝트에서는 모르는거 아니야」). 물음 3의 저량 · 유량 그림에 이 프로젝트가 보지 못한 것으로 적었다(3-나 ㉠). 같은 5번의 짐작(「그러면 lng감소는 석탄이나 유연하게 원자력이 가능하다면 그쪽에서 가져올 수 있겠지?」 — 1단계 결론 칸과 9단계는 「카타르 물량이 지정학 위험을 받으면 석탄 · 원전이 대신한다」로 적었다)은 석탄만 맞았다. 석탄은 2월부터 모델 예측보다 많았지만, 그것이 봉쇄 뒤 정부의 LNG 수요 관리 때문인지 계절관리제와 3월 16일 상한 해제 · 좁았던 값 차이 때문인지는 가를 수 없고(위 결정), 원전은 1 ~ 6월 내내 모델 예측보다 적어서 맞지 않았다.
3. 석탄을 더 올리지 않던 까닭(4단계 3-다 — 석탄 발전소가 43%를 더 만들 수 있었는데 만들지 않았다) — 평소에는 계절관리제 상한 같은 정책과 배출권 비용이 석탄을 줄이는 쪽이었다(9단계 4-가). 2026년에는 석탄의 몫이 1월부터 늘었고, 계절관리제가 끝난 것 · 3월 16일 상한 해제와 시기가 맞는다(판단). 보조금은 따로 찾지 않았다 — 발전 순서를 정하는 변동비의 연료단가에는 연료에 붙는 세금(관세 · 개별소비세 · 수입부과금 같은 것)이 들어 있고, 국내탄 발전기는 전력산업기반기금 지원금을 뺀 값을 쓴다(비용평가 세부운영규정 2.3.1.1.2 · 2.3.1.3.1.1 — 확인). 그래서 세금과 지원금은 2-가 값 차이에 이미 들어 있는 것으로 본다(판단 — 배출권 비용과 같은 까닭, 아래 9).
4. 가격 표는 세 해 평균보다 해마다 놓고 본다 — 평균은 2023년 하나에 끌려간다(창주 님 「이걸 평균과 비교하면 안될것 같어」).
5. 외부 자료는 확인 정도마다 쓸 수 있는 데까지만 쓴다 — 실측 · 확인은 사실로, 계획은 「하겠다고 했다」까지, 추정은 「그런 추정이 있다」로만(창주 님 「해석의 강도를 조절하는 방식이구나」).
6. 표를 읽는 말을 여섯으로 통일했다 — 실제 · 모델 예측 · 실제 − 모델 예측 · 평소 해 · 평소 범위 · 세 해 평균(표지 아래 「말 뜻」, 창주 님 「그냥 실제 - 모델예측 이런식으로 적어야지」).
7. 「그 밖」을 넷(수력 · 양수 · 유류 · 신재생 · 기타)으로, 신재생 · 기타를 태양광(역산 어림)과 나머지로 나눠 봤다 — 5 · 6월에 그 밖이 덜 만든 양의 대부분은 신재생 · 기타였고, 신재생 · 기타 가운데에서는 태양광보다 나머지(풍력 · 바이오 · 연료전지 같은 것)가 컸다. 양수는 대부분이 아니었다(창주 님 「질문 3은 넣어서 해석하는게 적합한거 같은데 지금 단계가 해석을 여러 측면에서 보는거니까」).
8. 석탄의 몫은 「석탄과 LNG가 더 만든 양 가운데 석탄의 몫」(2-아 ②)으로 적는다 — 전력 수요가 줄어든 몫은 석탄이 채우지 못한 양이 아니라 채울 필요가 없었던 양이라서. 6월은 이 몫을 낼 수 없어서(LNG도 덜 만들었다) 말로 적는다(창주 님 「2번이 더 적합한거 아닌가??」).
9. 배출권 비용을 넣은 값 차이는 따로 계산하지 않았다. 전력거래소 비용평가 세부운영규정(2025. 12.)에 따르면 변동비에 넣는 배출권 비용은 배출권 시장 값이 아니라, 발전사가 그 달 실제로 배출권을 사고판 돈을 그 달 태운 연료의 열량으로 나눈 값이고, 다음 달 열량단가에 넣는다(3 ~ 4쪽 · 31 ~ 32쪽). 발전사가 사고판 기록은 공개 자료에 없어서, 시장 값으로 다시 계산하면 규정과 다른 숫자가 나온다(창주 님 「좋아 그러면 하지 않았다면 하지않는 이유만 제대로 적히면 될 것 같어」).

**결정 근거**

- 2-가 [표 1] — 2026년 값 차이는 1 ~ 4월 +47.0 · +47.5 · +39.1 · +45.3으로 평소 해 셋보다 모두 좁았고(「2026년 − 세 해 평균」 −48.8 ~ −14.6), 5 · 6월은 +52.5 · +52.4로 평소 해(5월 +52.2 ~ +54.9, 6월 +47.8 ~ +53.2)와 비슷했다. 2-라 [표 1] — 평소 해 셋 모두 6월이 2월보다 쌌고, 2026년만 비쌌다(+17.4). 2-라 [표 2] — 석탄도 6월에 2월보다 비쌌다(+12.5).
- 2-마 「석탄이 대신 만든 비율」 — 1 ~ 4월 0 ~ 29%, 5월 92%, 6월 108%. 원전만 센 2-나는 5 · 6월 126% · 214%였다.
- 2-아 [표 1] 수요 칸 — 1 ~ 6월 −419 · −774 · −1,214 · −582 · +6 · −1,154(5월만 모델 예측과 거의 같았다). [표 2] ② — 1월 1% · 2월 36% · 3월 52% · 4월 50% · 5월 91%, 6월은 낼 수 없다. ①(2-마와 같은 비율)은 0 · 24 · 29 · 23 · 92 · 108%.
- 2-다 표 — 정부의 LNG 수요 관리(3월 16 · 24일, 4월 1 ~ 2일 — 확인 · 계획). LNG가 모자랐다는 근거는 찾지 못했다(추정뿐).
- 전력거래소 비용평가 세부운영규정(2025. 12., 창주 님이 받아 주신 파일 — `data/raw/`) — 3 ~ 4쪽 2.1.1 · 2.2.2: 열량단가는 연료 · 환경 · 배출권 열량단가를 합한 값이고, 배출권 열량단가는 매월 말 계산해 다음 달에 쓴다. 31 ~ 32쪽 2.3.4.1: 배출권 열량단가 = 그 달 배출권 순구매비용(산 돈 − 판 돈) ÷ 그 달 태운 연료의 열량. 공공데이터포털 「한국전력거래소_월간 연료비용 정보」 — 「비용평가위원회에서 의결한 발전비용 평가자료를 기준으로」 만든다.
- 2-바 [표 1] — 5월 그 밖 −608 가운데 신재생 · 기타 −374 · 양수 −107, 6월 −491 가운데 신재생 · 기타 −348 · 양수 −35. 2-사 [표 1] — 신재생 · 기타 가운데 태양광은 5월 −126 · 6월 −41, 나머지는 −249 · −307.
- 창주 님 — 2-마 아래 「✍️ 정할 것 — 물음 2의 결론 문장 (2)」 「좋아 …」 · 그 아래 「✍️ 다시 읽고 나서 — 짐작 ④ · ⑥」 · 「— 짐작 ⑤」 · 2-사 아래 「✍️ 정할 것 — 물음 2의 결론 문장 (3)」 「논리나 설명에 따르면 바꾼게 더 적합해보여」 · 2-아 아래 문답 「2번이 더 적합한거 아닌가??」 · 「해석이니까 디테일하게 적는게 맞겠다. 차라리 비유적인 표현 지우고 좀 딱딱하더라도 적는게 맞겠어」.

**남은 것**

- 물음 3(이 프로젝트가 본 것과 보지 못한 것의 경계) — 인과지도에 이 물음에서 본 것을 더한다. 「봉쇄 → LNG 가격」 근거는 「LNG 값은 5월부터 평소 해와 달랐다(봤다) · 봉쇄 때문인지는 판단」으로 적고, 가격을 거치지 않는 길(봉쇄 → 정부의 LNG 수요 관리 → 석탄 발전 상한 · 원전 정비 → 석탄 · 원전 → LNG)은 변수 하나와 화살표 셋으로 더하고, 석탄 설비용량은 뺀다. 클로드가 그려 두었고(`figures/인과지도_10단계`), 3-가에서 보신다. 짐작 ⑤ 「어디서 가져왔을까」는 저량 · 유량 그림(3-나)에 이 프로젝트가 보지 못한 것으로 적는다. → 10-29 물음 3에서 그렸다(📌 물음 3 결론).
- 11단계 한계(README 「한계」에도 적었다) —
  1. 석탄의 몫이 1월부터 늘어난 것과 시기가 맞는 까닭 셋(계절관리제와 3월 16일 상한 해제 · 1 ~ 4월에 좁았던 값 차이 · 정부의 LNG 수요 관리)은 데이터로 가를 수 없다. 9단계 4-나에서 상한이 석탄을 줄이고 있었는지도 가르지 못했다.
  2. 전력 수요가 줄었다는 것은 전력시장에서 거래된 양이 줄었다는 뜻이고, 전기를 덜 썼는지 전력시장 밖의 태양광이 채웠는지는 가를 수 없다.
  3. 6월에 석탄이 LNG 대신 만들었다는 것은 판단이다 — 같은 달에 석탄은 더 만들고 LNG는 덜 만들었다는 것까지가 숫자다.
  4. 그 밖의 나머지(풍력 · 바이오 · 연료전지 같은 것) 가운데 무엇이 5 · 6월에 덜 만들었는지는 2026년 자료로 나눌 수 없다.
  5. 2026년 태양광은 신재생 · 기타에서 그날 밤 값을 뺀 어림이라 실제보다 조금 크게 나온다(2024년에 3 ~ 10월 −0.7 ~ +7.8%, 11 ~ 2월 +8.3 ~ +13.0%). 태양광이 덜 만든 것이 서울 밖 날씨 때문인지 출력 제어 때문인지는 가를 수 없다.
  6. 2023년 연간 파일에는 소수력이 따로 없어서 수력에 들어 있을 수 있다 — 그렇다면 2-바의 수력 칸이 30일에 약 17 GWh 낮게, 신재생 · 기타 칸이 그만큼 높게 나온다(2024년 연간 파일의 소수력이 30일에 약 52 GWh이고, 모델이 배운 세 해 가운데 2023년에만 수력에 들어 있으니 모델 예측에 그 3분의 1쯤이 들어간다고 본 어림).
  7. 들여온 LNG(짐작 ⑤)는 이 프로젝트 자료에 없다.
  8. 2-다 ~ 2-아는 2-가 · 2-나의 결과를 본 뒤에 연 셀이다(2-바 · 2-사는 읽는 법을 결과 전에 적었고, 2-아는 검토가 2-마 표로 계산한 것을 보고 연 셀이다).
  9. 2-가 값 차이(EPSIS 연료비단가)에는 배출권 비용이 이미 들어 있는 것으로 본다(판단). 규정은 열량단가를 연료비 · 환경비 · 배출권 비용을 합한 값으로 정하고, 공공데이터포털은 이 자료를 비용평가 자료를 기준으로 만든다고 적었다. 다만 EPSIS 설명에 그렇게 적혀 있지는 않다. 들어 있더라도 배출권 비용은 발전사가 실제로 산 만큼만 한 달 늦게 들어가서, 2026년 6월 배출권 시장 값이 2025년의 세 배쯤으로 오른 것(아시아경제 · 한경ESG)이 그대로 들어간 것은 아니다. 어느 쪽이든 결정 줄은 그대로다 — 배출권 비용을 넣으면 석탄이 더 비싸진다. 까닭 후보 「1 ~ 4월에 좁았던 값 차이가 5월에 평소 해와 같아진 것」도 그대로 둔다. 값 차이는 4월 45.3에서 5월 52.5로 한 달에 7원 넘게 넓어졌는데, 배출권 비용이 이것을 뒤집으려면 한 달 사이에 석탄 쪽 배출권 비용이 그만큼 늘었어야 한다. 발전 부문은 2026년에도 배출권의 85%를 무상으로 받아서(계획 — 9단계 4-가 · 아시아경제) 그렇게 되기는 어렵다고 본다(판단).
  10. 석탄 발전소 정비는 보지 않았다. 4월에 정비하던 석탄 발전소가 5월에 돌아와 석탄을 더 쓸 수 있게 됐을 수도 있지만, 발전소마다의 정비 일정은 찾지 않았다. 받은 자료(전력거래소 연간 파일 · API)는 석탄을 연료별 합계로만 주고, 연간 파일의 석탄 설비용량은 한 해 내내 거의 같은 값이라(정비로 멈춘 발전소가 빠지지 않는다) 정비를 볼 수 없다. 그래서 정비는 까닭 후보로 결론에 넣지 않았다(2-아 아래 문답).
  11. 석탄의 몫은 2월에 이미 36%였다. 시기가 맞는 까닭 셋 가운데 2월에 걸리는 것은 계절관리제와 좁았던 값 차이인데, 둘 다 1월(몫 1%)에도 같았고(값 차이 1월 +47.0 · 2월 +47.5원/kWh — 2-가), 계절관리제는 석탄을 줄이는 쪽이다(9단계 4-가). 그래서 2월에 석탄의 몫이 늘어난 까닭은 이 자료로 설명되지 않는다(판단). 9단계 4-가에서도 석탄이 1월 22일 주부터 모델 예측보다 많아진 까닭을 외부 사건으로 찾지 못했다.

</div>


## 2-가

**이 셀이 답하는 물음** — 2026년 1 ~ 6월에 LNG가 석탄(유연탄)보다 비싼 정도가 평소 해보다 컸나?

**표의 숫자** — 달마다 LNG 연료비단가에서 석탄(유연탄) 연료비단가를 뺀 값, 곧 「LNG 값 − 석탄 값」(원/kWh)입니다. 연료비단가는 발전소가 전기 1 kWh를 만드는 데 쓴 연료의 값입니다. 전력거래소 규정은 이 값의 바탕인 열량단가를 연료비 · 환경비 · 배출권 비용을 합한 값으로 정하고 있어서, 배출권 비용도 들어 있는 것으로 봅니다(판단 — 결론 칸 「같이 정한 것」 9 · 한계 9). 유연탄은 석탄 발전소가 쓰는 석탄입니다. 무연탄도 석탄이지만 우리 자료에서 석탄 발전량의 1%가 안 돼서(2023년 0.94%, 2025년 0.73% — 전력거래소 연간 파일) 넣지 않았습니다. + 면 LNG가 석탄보다 비쌌다는 뜻이고, 맨 오른쪽 「2026년 − 세 해 평균」이 + 면 2026년에 LNG가 더 비쌌다는 뜻입니다. 세 해 평균은 같은 달 2023 ~ 2025년 값의 평균입니다(모델 없음). 모델끼리 비교하는 표가 아닙니다.

**가져온 자료** — 고르는 규칙: 인과지도의 「LNG 가격」(봉쇄 → LNG 가격 → 석탄 고리)에 들어오는 값을 찾았고, 기간은 모델이 배운 첫해인 2023년 1월부터 2026년 6월까지이며, 오른 달과 내린 달을 같이 봅니다(받은 파일에는 2026년 9월까지 있습니다). 확인 정도: 전력거래소 EPSIS 원본 — 확인. 우리 자료와 같은 대상인가: 한국 전력시장 발전소의 연료값이라 같습니다. 앞에서 정한 것: 가격 자료는 받아서 쓸지 이 물음에서 창주 님이 정하신다고 적어 두었고(표지 「앞 단계에서 넘어온 것」), 10-28에 정하셨습니다.

In [154]:
# 10단계 결과 해석 · 2-가
# ── EPSIS 「연료비용」의 연료비단가(원/kWh)로, 달마다 LNG가 유연탄보다 얼마나 비쌌나를 해마다 나란히 ──
파일 = os.path.join(RAW, "HOME_전력거래_연료비용.csv")                       # 10-28에 창주 님이 EPSIS에서 받은 파일
머리 = pd.read_csv(파일, encoding="cp949", header=None, nrows=3)              # 위 세 줄 = 값 이름 · 연료 이름 · 단위 (한글 윈도 글자로 적힌 파일)
assert 머리.iloc[0, 11] == "연료비단가" and 머리.iloc[1, 12] == "유연탄" and 머리.iloc[1, 15] == "LNG"   # 쓸 두 칸이 맞는 칸인지 확인
원본 = pd.read_csv(파일, encoding="cp949", header=None, skiprows=3)           # 넷째 줄부터가 달마다의 값
가격 = pd.DataFrame({"해": 원본[0].str[:4].astype(int),                        # 「2026/09」의 앞 네 글자 = 해
                     "달": 원본[0].str[5:7].astype(int),                       # 뒤 두 글자 = 달
                     "유연탄": 원본[12].astype(float),                         # 연료비단가 · 유연탄 (원/kWh)
                     "LNG": 원본[15].astype(float)})                           # 연료비단가 · LNG (원/kWh)
가격 = 가격[가격["달"] <= 6]                                                    # 9단계에서 비교한 1 ~ 6월만 본다
가격["차이"] = 가격["LNG"] - 가격["유연탄"]                                     # + 면 LNG가 유연탄보다 비쌌다

표 = 가격.pivot(index="달", columns="해", values="차이")                       # 달 × 해 표
세해평균 = 표[[2023, 2024, 2025]].mean(axis=1)                                 # 같은 달 2023 ~ 2025년의 평균(모델 없음)
표1 = pd.DataFrame({"2023년": 표[2023], "2024년": 표[2024], "2025년": 표[2025],
                    "세 해 평균": 세해평균, "2026년": 표[2026],
                    "2026년 − 세 해 평균 (+ 면 2026년에 LNG가 더 비쌌다)": 표[2026] - 세해평균})
표1.index = [f"{m}월" for m in 표1.index]                                       # 1 → 「1월」

print("[표 1] 달마다 LNG 값 − 석탄(유연탄) 값, 연료비단가 (원/kWh) — + 면 LNG가 석탄보다 비쌌다")
print("       연료비단가 = 발전소가 전기 1 kWh를 만드는 데 쓴 연료의 값 (EPSIS 「연료비용」)")
print()
display(표1.apply(lambda 열: 열.map(lambda v: f"{v:+.1f}")))                   # 소수 한 자리, + · − 를 붙여서

[표 1] 달마다 LNG 값 − 석탄(유연탄) 값, 연료비단가 (원/kWh) — + 면 LNG가 석탄보다 비쌌다
       연료비단가 = 발전소가 전기 1 kWh를 만드는 데 쓴 연료의 값 (EPSIS 「연료비용」)



,2023년,2024년,2025년,세 해 평균,2026년,2026년 − 세 해 평균 (+ 면 2026년에 LNG가 더 비쌌다)
1월,+132.2,+92.8,+62.2,+95.8,+47.0,-48.8
2월,+145.9,+78.5,+54.8,+93.1,+47.5,-45.6
3월,+113.5,+72.6,+49.7,+78.6,+39.1,-39.5
4월,+67.7,+59.1,+52.9,+59.9,+45.3,-14.6
5월,+54.9,+54.7,+52.2,+54.0,+52.5,-1.5
6월,+53.2,+52.3,+47.8,+51.1,+52.4,+1.3


## 2-나

**이 셀이 답하는 물음** — 2026년에 원전 발전량이 모델 예측보다 적었던 만큼을 석탄 발전량이 대신 만들었나?

**표의 숫자** — 모든 칸의 + · − 는 「실제 − 모델 예측」입니다. **+ 면 그 발전원의 실제 발전량이 모델 예측보다 많았다, − 면 적었다**는 뜻입니다. 모델 예측은 2023 ~ 2025년을 배운 선형회귀가 예측한 값입니다(9단계 2-나와 같다) — 2023 ~ 2025년을 배운 선형회귀가 날씨 · 시각 · 날 종류가 같을 때 맞힌 그 발전원의 발전량입니다. 30일 기준 GWh(한 시각 평균 × 720시각)로 적었고, 5월은 15일 평균입니다. 「석탄이 대신 만든 비율」은 석탄이 많았던 양을 원전이 적었던 양으로 나눈 값으로, 100%면 원전이 적었던 만큼 석탄이 많았다는 뜻입니다. 모델끼리 비교하는 표가 아닙니다. 1-다 [표 2]는 LNG가 늘어난 쪽을 + 로 적은 표라서, 원전 · 석탄의 부호가 이 표와 반대입니다.

In [155]:
# 10단계 결과 해석 · 2-나
# ── 2026년 달마다 원전 · 석탄 발전량이 모델 예측과 얼마나 달랐나 — 실제 − 모델 예측, 30일 기준 GWh (+ 면 실제가 모델 예측보다 많았다, − 면 적었다) ──
GWh = lambda mwh: mwh * 720 / 1000                                              # 한 시각 평균(MWh) × 720시각 ÷ 1,000 = 30일 기준(GWh)
원전차이 = -GWh(달몫["원전"])                                                     # 준비 ④의 달몫은 LNG가 늘어난 쪽이 + 라서 부호를 바꾼다 → 실제 − 모델 예측
석탄차이 = -GWh(달몫["석탄"])                                                     # 석탄도 같다 → 실제 − 모델 예측
대신비율 = 석탄차이 / -원전차이 * 100                                             # 석탄이 더 만든 양 ÷ 원전이 덜 만든 양 — 100% = 원전이 덜 만든 양만큼 석탄이 더 만들었다
표1 = pd.DataFrame({"원전 (실제 − 모델 예측, GWh)": 원전차이.map(lambda v: f"{v:+,.0f}"),
                    "석탄 (실제 − 모델 예측, GWh)": 석탄차이.map(lambda v: f"{v:+,.0f}"),
                    "석탄이 대신 만든 비율 = 석탄이 더 만든 양 ÷ 원전이 덜 만든 양": 대신비율.map(lambda v: f"{v:.0f}%")})
표1.index = [f"{m}월" for m in 표1.index]                                         # 1 → 「1월」

print("[표 1] 2026년 달마다 원전 · 석탄 발전량 — 실제 − 모델 예측, 30일 기준 GWh · + 면 실제가 모델 예측보다 많았다, − 면 적었다")
print("       모델 예측 = 2023 ~ 2025년을 배운 선형회귀가 그 날씨 · 시각 · 날 종류를 보고 예측한 발전량 · 5월은 15일 평균")
print()
display(표1)

[표 1] 2026년 달마다 원전 · 석탄 발전량 — 실제 − 모델 예측, 30일 기준 GWh · + 면 실제가 모델 예측보다 많았다, − 면 적었다
       모델 예측 = 2023 ~ 2025년을 배운 선형회귀가 그 날씨 · 시각 · 날 종류를 보고 예측한 발전량 · 5월은 15일 평균



,"원전 (실제 − 모델 예측, GWh)","석탄 (실제 − 모델 예측, GWh)",석탄이 대신 만든 비율 = 석탄이 더 만든 양 ÷ 원전이 덜 만든 양
1월,"-1,551",+5,0%
2월,"-2,625",+561,21%
3월,"-2,749",+804,29%
4월,"-1,059",+238,22%
5월,"-1,618","+2,039",126%
6월,-500,"+1,072",214%


## 2-다

**이 셀이 답하는 물음** — 2026년 3 ~ 6월에 LNG가 덜 들어왔나, 정부가 석탄 발전을 늘리게 했나?

**고르는 규칙** — 인과지도의 「호르무즈 봉쇄」에서 「석탄 발전」으로 가는 일 가운데 가격을 거치지 않는 것(LNG 도입 · 재고, 정부가 LNG를 아끼려고 석탄 · 원전을 더 쓰게 한 일)을 찾았습니다. 기간은 2026년 3 ~ 6월이고, 석탄을 늘리는 쪽과 줄이는 쪽을 같이 봤습니다. 한국의 자료만 씁니다. 9단계 4-가 표에 이미 있는 일(3월 16일 상한 해제)은 다시 찾지 않고 가리킵니다. 「LNG 수요 관리(정부)」는 지금 인과지도에 없는 변수라, 그리는 것은 물음 3(저량 · 유량 그림)에서 합니다. 2-나 결과를 본 뒤에 정한 규칙입니다.

**확인 정도** — 확인 = 정부 발표 원문이나 일어난 뒤에 쓴 기사 둘 이상 · 계획 = 하겠다는 발표 · 추정 = 분석 회사의 추정(기사 하나).

| 날짜 | 찾은 것 | 확인 정도 | 출처 |
|---|---|---|---|
| 3월 16 ~ 20일 | 정부가 LNG를 미리 관리하려고 석탄 · 원전을 한시적으로 더 쓰기로 했다 — 석탄 80% 상한 해제, 원전 정비를 5월 중순까지 앞당겨 원전 이용률을 80%대로 | 상한 해제는 확인(9단계 4-가 4번), 원전은 계획 | 9단계 4-가 · [쿠키뉴스 3-20](https://www.kukinews.com/article/view/kuk202603200137) |
| 3월 24일 | 국무회의 보고 — LNG 소비를 줄이려고 정비 중인 원전 5기를 5월까지 재가동하고, 미세먼지 상황에 따라 석탄발전도 탄력적으로 운영한다 | 계획(정부 보고를 전한 기사 둘) | [YTN 3-24](https://www.ytn.co.kr/_ln/0102_202603241057339813) · [산업일보 3-24](https://kidd.co.kr/news/245387) |
| 4월 1 ~ 2일 | 원유 자원안보 위기경보를 「경계」로, 천연가스를 「주의」로 올려 수요 관리를 강화했다 | 확인(정부 보도참고자료) | [보도참고자료 4-1](https://www.motir.go.kr/kor/article/ATCL3f49a5a8c/171679/view) |
| 4월 | 정부가 원전 이용률을 높이고 석탄 발전소 폐지를 미루는 쪽으로 가스 수요 관리를 강화하겠다고 했다. 6월 예정이던 석탄 3기(하동 1 · 보령 5 · 태안 2호기)의 폐쇄 일정도 다시 보기로 했다 | 계획(기사 둘) — 실제로 미뤘는지는 찾지 못함 | [ICIS 6-10](https://www.hellenicshippingnews.com/south-korea-lng-demand-may-rise-in-peak-summer-as-coal-headroom-narrows/) · [그린피스](https://www.greenpeace.org/korea/update/37082/blog-ce-assessment-energy-transition/) |
| 5 · 6월 | 한국 LNG 수입 5월 337만 톤 · 6월 326만 톤(예상) — 2025년 6월 348만 톤보다 조금 적다 | 추정(케이플러 — 기사 하나) | [더구루](https://www.theguru.co.kr/news/article.html?no=102887) |
| 5월 말 | 한국 LNG 재고 약 42% — 5년 평균 수준 | 추정(ICIS — 기사 하나) | [ICIS 6-10](https://www.hellenicshippingnews.com/south-korea-lng-demand-may-rise-in-peak-summer-as-coal-headroom-narrows/) |

**정부가 까닭으로 든 것** — 3월 24일 국무회의 보고는 「중동 사태 장기화와 자원안보위기 주의 경보 발령에 따른 에너지 절약 계획」이었습니다(YTN 3-24 — 정부 보고를 전한 기사). 4월 1일 보도참고자료는 원유 경보를 「경계」로 올린 까닭으로 「‘중동 전쟁’이 1개월 이상 장기화되고 원유 수급 차질, 국제 천연가스 가격 상승 등 위기가 가시화됨에 따라」와 「호르무즈 해협이 봉쇄되기 직전 마지막으로 통과한 유조선이 3월 20일 국내 입항한 이후, 열흘 넘게 호르무즈 발 원유 도입이 중단되면서」를 들었고, 천연가스 「주의」의 까닭으로 「동아시아 국제가격이 급등하여」를 들었습니다(확인 — 정부 보도참고자료 원문). 3-가 ② · 3-나 ㉤의 「정부 발표가 봉쇄를 까닭으로 들었다(확인)」는 이 4월 1일 자료입니다.

**찾지 못한 것** — 4 ~ 6월에 석탄 발전량을 늘리게 한 구체적인 조치(상한이나 발전 순서 규칙을 바꾼 일), 4월에 말한 석탄 폐지 연기를 실제로 했는지.

**표에 넣지 않은 것** — 같은 ICIS 기사의 「1 ~ 3월 석탄 발전량 +26.6%」는 2025년 같은 때와 비교한 값이라, 2023 ~ 2025년 세 해의 「평소」와 비교한 2-나와 기준이 달라 넣지 않았습니다.

## 2-라

**이 셀이 답하는 물음** — 봉쇄 뒤 LNG 값 자체가 올랐나, 올랐다면 언제부터인가? 석탄 값도 같이 봅니다.

**표의 숫자** — 연료비단가(발전소가 전기 1 kWh를 만드는 데 쓴 연료의 값, 원/kWh)가 그해 2월보다 얼마나 달라졌나입니다. + 면 2월보다 비싸졌다는 뜻입니다. 2월은 봉쇄(2월 28일 공격 · 3월 4일 봉쇄 선언 — 확인, 9단계 4-가 1 · 3번) 전 마지막 달이라 기준으로 삼았습니다. 세 해 평균은 같은 달 2023 ~ 2025년 값의 평균이고(모델 없음), 맨 오른쪽 「2026년 − 세 해 평균」이 + 면 2026년에 평소 해보다 더 비싸졌다는 뜻입니다. 해마다 값의 높이가 달라서(2023년은 다른 해보다 훨씬 비쌌다 — 2-가 답변), 높이가 아니라 2월에서 달라진 만큼을 해마다 나란히 놓았습니다. 9단계 4-나처럼 사건 앞뒤의 변화를 평소 해와 나란히 놓는 방식입니다. 모델끼리 비교하는 표가 아닙니다.

**값이 늦게 따라오는 까닭(외부 자료)** — 시장에서 거래하는 LNG의 연료단가는 두 달 전에 들어온 연료의 값으로 매기고([에너지경제신문 2020-10-22](https://m.ekn.kr/view.php?key=529698) — 기사 하나), 한국에 들여오는 LNG 값은 유가를 3 ~ 4개월쯤 늦게 따라간다고 합니다([메디게이트뉴스(아시아경제) 2020-08-06](https://medigatenews.com/stand/2661) — 기사 하나). 둘 다 값을 매기는 방식에 대한 설명이라 2026년 일을 적은 것은 아닙니다(확인 정도: 기사 하나씩).

In [156]:
# 10단계 결과 해석 · 2-라
# ── 봉쇄 전 마지막 달(2월)보다 연료비단가가 얼마나 달라졌나 — LNG · 석탄(유연탄)을 해마다 나란히 놓고 평소 해와 비교 ──
파일 = os.path.join(RAW, "HOME_전력거래_연료비용.csv")                         # 2-가와 같은 파일
머리 = pd.read_csv(파일, encoding="cp949", header=None, nrows=3)               # 위 세 줄 = 값 이름 · 연료 이름 · 단위
assert 머리.iloc[0, 11] == "연료비단가" and 머리.iloc[1, 12] == "유연탄" and 머리.iloc[1, 15] == "LNG"   # 쓸 두 칸이 맞는지 확인
원본 = pd.read_csv(파일, encoding="cp949", header=None, skiprows=3)            # 넷째 줄부터가 달마다의 값
연료값 = pd.DataFrame({"해": 원본[0].str[:4].astype(int),                       # 「2026/09」의 앞 네 글자 = 해
                       "달": 원본[0].str[5:7].astype(int),                      # 뒤 두 글자 = 달
                       "석탄": 원본[12].astype(float),                          # 연료비단가 · 유연탄 (원/kWh)
                       "LNG": 원본[15].astype(float)})                          # 연료비단가 · LNG (원/kWh)
연료값 = 연료값[연료값["달"] <= 6]                                               # 1 ~ 6월만

def 이월보다(연료):                                                               # 그달 값 − 그해 2월 값을 해마다 나란히
    표 = 연료값.pivot(index="달", columns="해", values=연료)                      # 달 × 해
    차 = (표 - 표.loc[2]).loc[[3, 4, 5, 6]]                                      # 해마다 2월을 0으로 놓고, 봉쇄 뒤 3 ~ 6월만
    세해평균 = 차[[2023, 2024, 2025]].mean(axis=1)                               # 같은 달 2023 ~ 2025년의 평균(모델 없음)
    나온 = pd.DataFrame({"2023년": 차[2023], "2024년": 차[2024], "2025년": 차[2025],
                         "세 해 평균": 세해평균, "2026년": 차[2026],
                         "2026년 − 세 해 평균 (+ 면 2026년에 더 비싸졌다)": 차[2026] - 세해평균})
    나온.index = [f"{m}월" for m in 나온.index]                                   # 3 → 「3월」
    return 나온.apply(lambda 열: 열.map(lambda v: f"{v:+.1f}"))                  # 소수 한 자리, + · − 를 붙여서

print("[표 1] LNG 연료비단가가 그해 2월보다 얼마나 달라졌나 (원/kWh) — + 면 2월보다 비싸졌다")
print()
display(이월보다("LNG"))
print()
print("[표 2] 석탄(유연탄) 연료비단가가 그해 2월보다 얼마나 달라졌나 (원/kWh) — + 면 2월보다 비싸졌다")
print()
display(이월보다("석탄"))

[표 1] LNG 연료비단가가 그해 2월보다 얼마나 달라졌나 (원/kWh) — + 면 2월보다 비싸졌다



,2023년,2024년,2025년,세 해 평균,2026년,2026년 − 세 해 평균 (+ 면 2026년에 더 비싸졌다)
3월,-38.7,-1.2,-3.1,-14.3,-3.9,+10.4
4월,-88.0,-13.4,+2.7,-32.9,+1.6,+34.5
5월,-102.3,-16.7,-1.8,-40.3,+8.6,+48.9
6월,-107.5,-19.3,-5.5,-44.1,+17.4,+61.5



[표 2] 석탄(유연탄) 연료비단가가 그해 2월보다 얼마나 달라졌나 (원/kWh) — + 면 2월보다 비싸졌다



,2023년,2024년,2025년,세 해 평균,2026년,2026년 − 세 해 평균 (+ 면 2026년에 더 비싸졌다)
3월,-6.3,+4.8,+2.0,+0.1,+4.5,+4.3
4월,-9.7,+6.0,+4.6,+0.3,+3.8,+3.5
5월,-11.3,+7.1,+0.7,-1.2,+3.6,+4.7
6월,-14.7,+6.9,+1.5,-2.1,+12.5,+14.6


## 2-마

**이 셀이 답하는 물음** — 석탄보다 싼 발전원(원전과 그 밖)이 모델 예측보다 적었던 만큼을, 석탄이 달마다 얼마나 대신 만들었나?

**표의 숫자** — 모든 칸은 「실제 − 모델 예측」(+ 면 실제가 모델 예측보다 많았다, − 면 적었다)이고, 30일 기준 GWh입니다. 모델 예측은 2023 ~ 2025년을 배운 선형회귀가 예측한 값입니다(9단계 2-나와 같다). 「그 밖」은 원전 · 석탄 · LNG를 뺀 발전원으로, 대부분 태양광 · 수력 · 풍력처럼 연료비가 거의 없는 발전원이고 양수 · 유류도 조금 들어 있습니다(유류는 LNG보다 비싸서 엄밀하게는 「석탄보다 싼」에 들지 않지만 양이 적습니다 — 판단). 「둘을 더한 값」이 − 면 석탄보다 싼 발전원이 그만큼 비었다는 뜻이고, 「석탄이 대신 만든 비율」은 석탄이 늘어난 양을 덜 만든 양으로 나눈 값입니다(100%면 다 대신 만들었다). 덜 만든 양 가운데 석탄이 대신 만들지 않은 나머지는 LNG가 더 만들었거나 수요가 모델 예측보다 적어서 필요 없었습니다 — 그래서 LNG 칸을 같이 놓았습니다. 숫자는 1-다 [표 2]의 원전 · 그 밖 · 석탄 줄과 같은 값을 「실제 − 모델 예측」으로 바꿔 적은 것이라, 1-다 [표 2]와 부호가 반대입니다. 모델끼리 비교하는 표가 아닙니다.

In [157]:
# 10단계 결과 해석 · 2-마
# ── 석탄보다 싼 발전원(원전 · 그 밖)이 모델 예측보다 덜 만든 양을 석탄이 얼마나 대신 만들었나 — 실제 − 모델 예측, 30일 기준 GWh ──
GWh = lambda mwh: mwh * 720 / 1000                                              # 한 시각 평균(MWh) × 720시각 ÷ 1,000 = 30일 기준(GWh)
원전차이 = -GWh(달몫["원전"])                                                     # 준비 ④의 달몫은 LNG가 늘어난 쪽이 + 라서 부호를 바꾼다 → 실제 − 모델 예측
그밖차이 = -GWh(달몫["그 밖"])                                                    # 그 밖(태양광 · 수력 · 풍력 같은 것과 양수 · 유류)도 같다 → 실제 − 모델 예측
덜만든양 = 원전차이 + 그밖차이                                                    # 둘을 더한 값 — − 면 석탄보다 싼 발전원이 모델 예측보다 그만큼 덜 만들었다
석탄차이 = -GWh(달몫["석탄"])                                                     # 석탄 → 실제 − 모델 예측
LNG차이 = GWh(달몫["선형회귀"])                                                   # 선형회귀가 본 LNG — 이미 + 면 LNG가 많았다(실제 − 모델 예측)
대신비율 = 석탄차이 / -덜만든양 * 100                                             # 석탄이 더 만든 양 ÷ 원전과 그 밖이 덜 만든 양 — 100% = 덜 만든 양을 석탄이 다 대신 만들었다
쉼표 = lambda v: f"{v:+,.0f}"                                                     # 천 단위 쉼표, + · − 를 붙여서
표1 = pd.DataFrame({"원전": 원전차이.map(쉼표), "그 밖": 그밖차이.map(쉼표),
                    "둘을 더한 값 (− 면 원전과 그 밖이 모델 예측보다 덜 만들었다)": 덜만든양.map(쉼표),
                    "석탄": 석탄차이.map(쉼표),
                    "석탄이 대신 만든 비율 = 석탄 ÷ 원전과 그 밖이 덜 만든 양": 대신비율.map(lambda v: f"{v:.0f}%"),
                    "LNG (선형회귀)": LNG차이.map(쉼표)})
표1.index = [f"{m}월" for m in 표1.index]                                         # 1 → 「1월」

print("[표 1] 2026년 달마다 발전원별 발전량 — 실제 − 모델 예측, 30일 기준 GWh · + 면 실제가 모델 예측보다 많았다, − 면 적었다 (5월은 15일 평균)")
print("       모델 예측 = 9단계 2-나와 같은 선형회귀의 예측 · 원전과 그 밖이 덜 만든 양 가운데 석탄이 대신 만들지 않은 나머지는, LNG가 더 만들었거나 수요가 모델 예측보다 적어서 필요 없었다")
print()
display(표1)

[표 1] 2026년 달마다 발전원별 발전량 — 실제 − 모델 예측, 30일 기준 GWh · + 면 실제가 모델 예측보다 많았다, − 면 적었다 (5월은 15일 평균)
       모델 예측 = 9단계 2-나와 같은 선형회귀의 예측 · 원전과 그 밖이 덜 만든 양 가운데 석탄이 대신 만들지 않은 나머지는, LNG가 더 만들었거나 수요가 모델 예측보다 적어서 필요 없었다



,원전,그 밖,둘을 더한 값 (− 면 원전과 그 밖이 모델 예측보다 덜 만들었다),석탄,석탄이 대신 만든 비율 = 석탄 ÷ 원전과 그 밖이 덜 만든 양,LNG (선형회귀)
1월,"-1,551",+319,"-1,232",+5,0%,+809
2월,"-2,625",+282,"-2,343",+561,24%,"+1,008"
3월,"-2,749",-15,"-2,764",+804,29%,+746
4월,"-1,059",+4,"-1,055",+238,23%,+235
5월,"-1,618",-608,"-2,226","+2,039",92%,+194
6월,-500,-491,-991,"+1,072",108%,"-1,235"


## 2-바

**이 셀이 답하는 물음** — 5 · 6월에 「그 밖」 발전원이 모델 예측보다 덜 만들었는데, 그중 무엇이 덜 만들었나?

**표의 숫자** — 모든 칸은 「실제 − 모델 예측」(+ 면 실제가 모델 예측보다 많았다, − 면 적었다)이고, 30일 기준 GWh입니다(5월은 15일 평균). 모델 예측은 2-마와 같은 선형회귀(2023 ~ 2025년을 배움)가 그 발전원 하나만 맞힌 값입니다. 넷을 더한 값은 2-마의 「그 밖」 칸과 같습니다(표 아래 확인 줄). 모델끼리 비교하는 표가 아닙니다.

**넷으로 맞춘 방법** — 2026년 자료(API)는 그 밖을 수력 · 양수 · 유류 · 신재생 · 기타 넷으로 줍니다. 2023 · 2025년 연간 파일은 연료를 더 잘게 주어서 넷에 맞춰 묶었습니다 — 수력은 「수력」, 양수는 「양수」, 유류는 「중유 + 경유」, 나머지(태양광 · 풍력 · 바이오 · 연료전지 · 소수력 · IGCC · 부생가스 같은 것)는 신재생 · 기타입니다. 2024년은 두 자료가 겹쳐서 이렇게 묶는 것이 맞는지 봤습니다 — 수력은 두 자료가 같았고(소수력은 들어 있지 않다), IGCC는 API에서 신재생 · 기타에 들어 있었습니다. 2023년 연간 파일에는 소수력 줄이 따로 없어서 수력에 들어 있을 수 있습니다. 그렇다면 수력 칸이 30일에 약 17 GWh 낮게, 신재생 · 기타 칸이 그만큼 높게 나옵니다 — 2024년 연간 파일의 소수력이 30일에 약 52 GWh이고, 모델이 배운 세 해 가운데 2023년에만 수력에 들어 있으니 모델 예측에 그 3분의 1쯤이 들어간다고 본 어림입니다.

In [158]:
# 10단계 결과 해석 · 2-바
# ── 「그 밖」을 넷(수력 · 양수 · 유류 · 신재생 · 기타)으로 나눠 본다 — 실제 − 모델 예측, 30일 기준 GWh ──
넷이름 = ["수력", "양수", "유류", "신재생 · 기타"]                                   # 2026년 자료(API)가 그 밖을 주는 넷
바꿈 = {"수력": "수력", "양수": "양수", "유전": "유류", "신재생·기타": "신재생 · 기타"}   # API의 연료 이름 → 이 표의 이름
def 연간넷(해):                                                                       # 전력거래소 연간 파일 한 해 — 그 밖을 API의 넷에 맞춰 묶는다
    이름 = [f for f in os.listdir(os.path.join(RAW, "kpx")) if f"{해}1231" in f][0]
    d = pd.read_csv(os.path.join(RAW, "kpx", 이름), encoding="utf-8-sig")
    d.columns = ["date", "hour", "fuel", "cap_mw", "mwh"]                              # 준비 ④와 같다
    d["fuel"] = d["fuel"].str.strip()
    d = d[~d["fuel"].isin(["LNG", "원자력"] + 석탄종류)].copy()                        # LNG · 원전 · 석탄을 뺀 나머지가 그 밖 — LNG 줄은 여기서 버린다
    d["묶음"] = d["fuel"].map({"수력": "수력", "양수": "양수", "중유": "유류", "경유": "유류"}).fillna("신재생 · 기타")   # 나머지는 모두 신재생 · 기타
    d["ts"] = pd.to_datetime(d["date"]) + pd.to_timedelta(d["hour"].astype(int) - 1, unit="h")   # 1 ~ 24시를 0 ~ 23시 시각으로
    return d.pivot_table(index="ts", columns="묶음", values="mwh", aggfunc="sum").fillna(0)
넷 = pd.concat([연간넷(2023), a24.rename(columns=바꿈)[넷이름], 연간넷(2025)]).sort_index().reindex(전체.index)[넷이름]   # 2023 ~ 2025년 시각마다 넷
넷26 = a26.rename(columns=바꿈)[넷이름]                                              # 2026년 — 준비 ④에서 이미 2026년 시각에 맞춰 두었다
assert 넷.notna().all().all() and 넷26.notna().all().all(), "빈 시각이 있다"

GWh = lambda mwh: mwh * 720 / 1000                                                   # 한 시각 평균(MWh) × 720시각 ÷ 1,000 = 30일 기준(GWh)
차이넷 = pd.DataFrame({k: GWh((넷26[k] - 평소라면값(넷[k])).groupby(넷26.index.month).mean()) for k in 넷이름})   # 실제 − 모델 예측, 달마다
차이넷["넷을 더한 값"] = 차이넷[넷이름].sum(axis=1)                                  # 2-마의 「그 밖」과 같아야 한다
어긋남 = (차이넷["넷을 더한 값"] - (-GWh(달몫["그 밖"]))).abs().max()                # 2-마 「그 밖」(준비 ④의 달몫은 부호가 반대)과 비교

쉼표 = lambda v: f"{v:+,.0f}"                                                        # 천 단위 쉼표, + · − 를 붙여서
표1 = 차이넷.apply(lambda s: s.map(쉼표))
표1.index = [f"{m}월" for m in 표1.index]                                            # 1 → 「1월」
print("[표 1] 2026년 달마다 「그 밖」 발전원별 발전량 — 실제 − 모델 예측, 30일 기준 GWh · + 면 실제가 모델 예측보다 많았다, − 면 적었다 (5월은 15일 평균)")
print("       모델 예측 = 2-마와 같은 선형회귀가 그 발전원 하나만 맞힌 값 · 신재생 · 기타 = 태양광 · 풍력 · 바이오 · 연료전지 · 소수력 같은 것(태양광만 따로는 없다)")
print()
display(표1)
print(f"넷을 더한 값과 2-마의 「그 밖」이 가장 크게 어긋난 달 {어긋남:.1f} GWh (0이어야 한다)")

[표 1] 2026년 달마다 「그 밖」 발전원별 발전량 — 실제 − 모델 예측, 30일 기준 GWh · + 면 실제가 모델 예측보다 많았다, − 면 적었다 (5월은 15일 평균)
       모델 예측 = 2-마와 같은 선형회귀가 그 발전원 하나만 맞힌 값 · 신재생 · 기타 = 태양광 · 풍력 · 바이오 · 연료전지 · 소수력 같은 것(태양광만 따로는 없다)



,수력,양수,유류,신재생 · 기타,넷을 더한 값
1월,-9,-68,-52,+448,+319
2월,-5,-71,-39,+396,+282
3월,-27,-123,-32,+166,-15
4월,-103,-122,-19,+247,+4
5월,-112,-107,-15,-374,-608
6월,-97,-35,-11,-348,-491


넷을 더한 값과 2-마의 「그 밖」이 가장 크게 어긋난 달 0.0 GWh (0이어야 한다)


## 2-사

**이 셀이 답하는 물음** — 5 · 6월에 신재생 · 기타가 모델 예측보다 덜 만든 양 가운데, 태양광은 얼마이고 나머지(풍력 · 바이오 · 연료전지 같은 것)는 얼마인가? (창주 님 「태양광 따로 없으면 앞의 뭐 태양광이나 일사 데이터와 연결해서 못보는거 아니야 그러면?」)

**어떻게 뽑나** — 밤에는 태양광이 0이라, 그날 1 ~ 5시 신재생 · 기타의 평균을 태양광이 아닌 것으로 보고, 시각마다 「신재생 · 기타 − 그날 1 ~ 5시 평균」을 태양광으로 어림합니다(0보다 작으면 0). 2024년 실제와 맞춰 보면 3 ~ 10월은 −0.7 ~ +7.8%, 11 ~ 2월은 +8.3 ~ +13.0% 크게 나왔습니다(9단계 1-바). 모델 예측은 2023 ~ 2025년의 실제 태양광 발전량을 2-마와 같은 선형회귀로 배워 맞힌 값입니다.

**표의 숫자** — 모든 칸은 「실제 − 모델 예측」(+ 면 실제가 모델 예측보다 많았다, − 면 적었다)이고, 30일 기준 GWh입니다(5월은 15일 평균). 2026년 태양광의 「실제」는 역산 어림입니다. 「나머지」는 신재생 · 기타에서 태양광을 뺀 값이라, 태양광 + 나머지 = 신재생 · 기타입니다. 모델끼리 비교하는 표가 아닙니다.

In [159]:
# 10단계 결과 해석 · 2-사
# ── 신재생 · 기타를 태양광(역산 어림)과 나머지로 나눈다 — 실제 − 모델 예측, 30일 기준 GWh ──
신재26 = 넷26["신재생 · 기타"]                                                       # 2026년 신재생 · 기타 — 2-바와 같다
밤시각 = 신재26.index.hour.isin([0, 1, 2, 3, 4])                                     # 1 ~ 5시(0 ~ 4시 시각) — 태양광이 0인 때
밤 = 신재26[밤시각].groupby(신재26.index[밤시각].normalize()).mean()                 # 그날 밤 평균 — 9단계 1-바 · 2-나와 같다
태양광26 = (신재26 - 밤.reindex(신재26.index.normalize()).values).clip(lower=0)     # 신재생 · 기타 − 그날 밤 평균, 0보다 작으면 0 — 역산 어림
태양광실제 = 전체["solar_mwh"]                                                       # 2023 ~ 2025년은 실제 태양광 발전량이 있다(9단계 2-나와 같다)
assert 태양광실제.notna().all() and 태양광26.notna().all(), "빈 시각이 있다"
태양광차이 = GWh((태양광26 - 평소라면값(태양광실제)).groupby(태양광26.index.month).mean())   # 실제(역산 어림) − 모델 예측, 달마다

표 = pd.DataFrame({"신재생 · 기타 (2-바)": 차이넷["신재생 · 기타"], "그 가운데 태양광 (역산 어림)": 태양광차이})
표["태양광을 뺀 나머지 (풍력 · 바이오 · 연료전지 같은 것)"] = 표["신재생 · 기타 (2-바)"] - 표["그 가운데 태양광 (역산 어림)"]   # 태양광 + 나머지 = 신재생 · 기타
표1 = 표.apply(lambda s: s.map(쉼표))
표1.index = [f"{m}월" for m in 표1.index]                                            # 1 → 「1월」
print("[표 1] 2026년 달마다 신재생 · 기타를 태양광과 나머지로 나눔 — 실제 − 모델 예측, 30일 기준 GWh · + 면 실제가 모델 예측보다 많았다, − 면 적었다 (5월은 15일 평균)")
print("       2026년 태양광 = 신재생 · 기타 − 그날 1 ~ 5시 평균(역산 어림 — 9단계 1-바) · 모델 예측 = 2023 ~ 2025년 실제 태양광을 2-마와 같은 선형회귀로 배워 맞힌 값")
print()
display(표1)

[표 1] 2026년 달마다 신재생 · 기타를 태양광과 나머지로 나눔 — 실제 − 모델 예측, 30일 기준 GWh · + 면 실제가 모델 예측보다 많았다, − 면 적었다 (5월은 15일 평균)
       2026년 태양광 = 신재생 · 기타 − 그날 1 ~ 5시 평균(역산 어림 — 9단계 1-바) · 모델 예측 = 2023 ~ 2025년 실제 태양광을 2-마와 같은 선형회귀로 배워 맞힌 값



,신재생 · 기타 (2-바),그 가운데 태양광 (역산 어림),태양광을 뺀 나머지 (풍력 · 바이오 · 연료전지 같은 것)
1월,+448,+272,+176
2월,+396,+291,+106
3월,+166,+298,-131
4월,+247,+153,+95
5월,-374,-126,-249
6월,-348,-41,-307


## 2-아

**이 셀이 답하는 물음** — 원전과 그 밖이 모델 예측보다 덜 만든 양 가운데, 석탄과 LNG가 더 만든 양은 얼마이고, 수요가 줄어 채울 필요가 없었던 양은 얼마인가? 석탄과 LNG만 놓고 보면, 석탄의 몫은 달마다 얼마인가?

**표의 숫자** — [표 1]은 모든 칸이 「실제 − 모델 예측」이고(+ 면 실제가 모델 예측보다 많았다, − 면 적었다), 30일 기준 GWh입니다(5월은 15일 평균). 수요 = 원전 + 그 밖 + 석탄 + LNG이고, 전력시장에서 거래된 모든 발전량의 합입니다. [표 2]는 석탄의 몫 두 가지(%)입니다 — ① 석탄 ÷ 원전과 그 밖이 덜 만든 양(2-마와 같다), ② 석탄 ÷ (석탄 + LNG가 더 만든 양). ②는 수요가 줄어 채울 필요가 없었던 양을 빼고, 더 만든 쪽(석탄과 LNG)만 놓고 석탄의 몫을 본 것입니다. LNG도 모델 예측보다 덜 만든 달은 ②를 내지 않습니다. 모델끼리 비교하는 표가 아닙니다.

In [160]:
# 10단계 결과 해석 · 2-아
# ── 원전 · 그 밖 · 석탄 · LNG와 그 넷을 더한 수요 — 실제 − 모델 예측, 30일 기준 GWh · 석탄의 몫 두 가지 ──
GWh = lambda mwh: mwh * 720 / 1000                                             # 한 시각 평균(MWh) × 720시각 ÷ 1,000 = 30일 기준(GWh)
차 = pd.DataFrame({"원전": -GWh(달몫["원전"]),                                    # 준비 ④의 달몫은 LNG가 늘어난 쪽이 + 라서 부호를 바꾼다 → 실제 − 모델 예측
                   "그 밖": -GWh(달몫["그 밖"]),
                   "석탄": -GWh(달몫["석탄"]),
                   "LNG (선형회귀)": GWh(달몫["선형회귀"])})                       # LNG는 이미 실제 − 모델 예측
차["수요 = 넷을 더한 값"] = 차[["원전", "그 밖", "석탄", "LNG (선형회귀)"]].sum(axis=1)   # 모든 발전원의 합 = 전력 수요(전력시장에서 거래된 양)
어긋남 = (차["수요 = 넷을 더한 값"] - GWh(달몫["수요"])).abs().max()             # 준비 ④가 따로 맞힌 수요와 같아야 한다

덜만든양 = -(차["원전"] + 차["그 밖"])                                            # 원전과 그 밖이 모델 예측보다 덜 만든 양 (2-마의 둘을 더한 값의 크기)
몫1 = 차["석탄"] / 덜만든양 * 100                                                 # 2-마와 같은 비율 — 석탄 ÷ 원전과 그 밖이 덜 만든 양
더만든양 = 차["석탄"] + 차["LNG (선형회귀)"]                                      # 석탄과 LNG가 모델 예측보다 더 만든 양
몫2 = (차["석탄"] / 더만든양 * 100).where(차["LNG (선형회귀)"] >= 0)              # 석탄 ÷ (석탄 + LNG) — LNG도 덜 만든 달은 내지 않는다

쉼표 = lambda v: f"{v:+,.0f}"                                                     # 천 단위 쉼표, + · − 를 붙여서
표1 = 차.apply(lambda s: s.map(쉼표))
표1.index = [f"{m}월" for m in 표1.index]
표2 = pd.DataFrame({"① 석탄 ÷ 원전과 그 밖이 덜 만든 양 (2-마)": 몫1.map(lambda v: f"{v:.0f}%"),
                    "② 석탄 ÷ (석탄 + LNG가 더 만든 양)": 몫2.map(lambda v: "— (LNG도 덜 만들었다)" if pd.isna(v) else f"{v:.0f}%")})
표2.index = 표1.index

print("[표 1] 2026년 달마다 발전량 — 실제 − 모델 예측, 30일 기준 GWh · + 면 실제가 모델 예측보다 많았다, − 면 적었다 (5월은 15일 평균)")
print("       수요 = 원전 + 그 밖 + 석탄 + LNG · 수요가 − 인 달은 모든 발전원을 합친 양이 모델 예측보다 적었다")
display(표1)
print(f"수요 칸과 준비 ④가 따로 맞힌 수요가 가장 크게 어긋난 달 {어긋남:.1f} GWh (0이어야 한다)")
print()
print("[표 2] 석탄의 몫 두 가지 (%) — ①은 2-마와 같고, ②는 수요가 줄어 채울 필요가 없었던 양을 빼고 석탄과 LNG만 놓고 본 것")
display(표2)

[표 1] 2026년 달마다 발전량 — 실제 − 모델 예측, 30일 기준 GWh · + 면 실제가 모델 예측보다 많았다, − 면 적었다 (5월은 15일 평균)
       수요 = 원전 + 그 밖 + 석탄 + LNG · 수요가 − 인 달은 모든 발전원을 합친 양이 모델 예측보다 적었다


,원전,그 밖,석탄,LNG (선형회귀),수요 = 넷을 더한 값
1월,"-1,551",+319,+5,+809,-419
2월,"-2,625",+282,+561,"+1,008",-774
3월,"-2,749",-15,+804,+746,"-1,214"
4월,"-1,059",+4,+238,+235,-582
5월,"-1,618",-608,"+2,039",+194,+6
6월,-500,-491,"+1,072","-1,235","-1,154"


수요 칸과 준비 ④가 따로 맞힌 수요가 가장 크게 어긋난 달 0.0 GWh (0이어야 한다)

[표 2] 석탄의 몫 두 가지 (%) — ①은 2-마와 같고, ②는 수요가 줄어 채울 필요가 없었던 양을 빼고 석탄과 LNG만 놓고 본 것


,① 석탄 ÷ 원전과 그 밖이 덜 만든 양 (2-마),② 석탄 ÷ (석탄 + LNG가 더 만든 양)
1월,0%,1%
2월,24%,36%
3월,29%,52%
4월,23%,50%
5월,92%,91%
6월,108%,— (LNG도 덜 만들었다)


# 물음 3 — 이 프로젝트가 본 것과 보지 못한 것의 경계는 어디인가

<div style="border: 2px solid var(--vscode-focusBorder, #3794ff); border-left-width: 6px; border-radius: 8px; padding: 4px 18px 10px; background: var(--vscode-textBlockQuote-background, rgba(127, 127, 127, 0.1));">

## 📌 물음 3 결론

**왜 생긴 물음인가** — 물음 1 · 2에서 9단계의 차이를 배 몇 척으로 바꾸고 석탄이 많았던 까닭을 찾은 뒤, 이 프로젝트가 본 것과 보지 못한 것의 경계를 그림으로 밝히는 물음입니다(지침 10단계 칸 · 「■ 몸통」 시스템 다이내믹스 줄). 창주 님이 이 물음이 정하는 것을 「…추가할 것들은 추가하고, 한계점으로 인과지도에 적힌 것중 못 넣는 것들은 표시하고 하겠다는거잖어??」로 적으셨습니다.

### 결정 — LNG가 들어오고 쌓이고 쓰이는 양 가운데 이 프로젝트가 본 것은 LNG 발전량 하나다. 2026년에 모델 예측과 달랐던 원전 · 석탄 · 전력 수요와 그 까닭 후보는 모두 모델 밖에 있고, 재고와 들여온 양은 보지 못해 LNG가 모자랐는지는 말할 수 없다

**창주 님이 정한 결론 문장** — 「LNG가 들어오고 쌓이고 쓰이는 양 가운데 이 프로젝트가 본 건 LNG 발전량 하나야. 모델이 보는 것(몇 시인가, 무슨 날인가, 기온, 햇빛, 몇 월인가)에는 2026년에 달랐던 원전, 석탄, 전력 수요와 그 까닭 후보인 정부의 LNG 수요 관리가 없으니까, 그건 다 실제 − 모델 예측에 섞여 있는 거지. 그리고 재고랑 들여온 양은 모르니까 2026년에 LNG가 모자랐는지는 알 수 없어. 물음 1의 배 몇 척도 발전에 쓴 양을 바꾼 거지, 실제로 들어온 배가 아니고.」(「✍️ 정할 것」 1 아래 「✍️ 다시 읽고 나서」 — 3-가 · 3-나에서 하신 말로 클로드가 엮은 초안을 고치신 글이고, 마지막 문장은 3-나 문답에서 클로드가 답한 내용)

**같이 정한 것**

1. 11단계 README에 넣을 그림은 둘 다 — 10단계 인과지도와 저량 · 유량 그림(창주 님 「두개가 보여주는게 다르니까 둘 다 적는 다 방향이 맞는거 같어.」). 인과지도는 모델이 보는 것과 보지 못하는 것을, 저량 · 유량 그림은 자료가 본 것과 보지 못한 것을 보여 준다. 인과지도를 10단계 그림으로 하는 것은 9단계를 닫을 때 정하신 「최신 버전」대로다. 물음 3 머리 칸 틀의 「그림 한 장」에서 하나 늘었고, 「다」는 채팅에서 물으셔서 클로드가 먼저 드린 판단이기도 하다(「✍️ 정할 것」 아래 답변).
2. 인과지도(3-가) — 9단계 그림에 정부의 LNG 수요 관리와 화살표 셋을 더하고 석탄 설비용량과 그 화살표를 뺐다. 화살표 여덟에 근거(봤다 · 확인 · 계획 · 판단 · 2026년엔 아님 · 못 봄)를 달았다(클로드가 그림 — 클로드 판단). 「2026년엔 아님」은 자료로 보니 2026년에는 그 화살표대로 움직이지 않은 것(⑧), 「못 봄」은 자료가 없어 확인하지 못한 것이다 — 문답 때는 둘 다 「못 봄」이었는데, 창주 님 말(「그러면 못봄이라는거 두 가지 표현으로 쓰면 헷갈리는거 아니야??」)로 10단계를 닫을 때 갈랐다. ⑤는 「못 봄」이다 — 우리 자료의 LNG 값은 두 달쯤 늦게 따라와 3월 16일 상한 해제 때의 시장 값을 볼 수 없다(3-가 「못 봄 확인 (4)」, 창주 님 「그것은 가격에 대한 데이터가 있어야 하지 않나?」).
3. 저량 · 유량 그림(3-나) — 보관 비용 화살표는 그리지 않고 「재고 목표(비축 의무와 겨울 대비)」를 넣었다. 7단계에서 물으신 「lng 보관할 때 초저온으로 보관해야하는데 거기에 유지비용에 따라서 판단해서 lng수요가 줄수도 있나??」는 한국에서는 맞지 않다 — 가스공사는 비축 의무(하루 평균 판매량의 9일분 — 확인)와 겨울 대비 목표(계획, 기사 하나)에 맞춰 재고를 둔다(클로드 판단 — 독립 검토 둘, 3-나 보관 비용 확인 아래 답변).

**이 물음에서 알게 된 것** — 2026년 1 ~ 6월에 모델 예측과 달랐던 것(원전 · 석탄 · 전력 수요)과 그 까닭 후보(정부의 LNG 수요 관리)는 모두 모델이 보는 변수 밖에 있었습니다(3-가). LNG가 들어오고 쌓이고 쓰이는 양 가운데 이 프로젝트가 자료로 본 것은 LNG 발전량 하나라서, LNG 재고나 들여온 양이 어땠는지 — LNG가 모자랐는지 — 는 말할 수 없습니다(3-나). 7단계부터 이어 온 「보관 비용이 들여오는 양을 줄이나」는 한국에서는 맞지 않았습니다.

**결정 근거**

- 3-가 — 그림의 굵은 글자 여섯(모델이 보는 변수)과 문답 다섯. 경계 확인에서 창주 님 「아니 여기 6개에서는 그 물음 2와 다른것과 연결지을건 없어보여」.
- 3-나 — 그림의 초록 글자(LNG 발전량)와 문답 넷. 경계 확인에서 창주 님 「알수 없어. 재고를 모르니까.」 · 보관 비용 확인 아래 답변(비축 의무 — 확인 · 겨울 대비 목표 — 계획, 기사 하나 · 장기 계약 70 ~ 80% — 기사 하나).
- 창주 님 — 「✍️ 정할 것」 1 · 2와 그 아래 「✍️ 다시 읽고 나서」(고친 결론 문장).

**남은 것**

- 11단계(결론 — README를 쓰는 단계)에서 README에 두 그림과 경계 한 문단을 넣는다. 경계 문단은 위 결론 문장이 바탕이다(닫는 칸 ②).
- 2026년에 LNG가 모자랐는지와 재고가 버틴 뒤 7월 이후에 나타났을 영향은 이 프로젝트로 말할 수 없다 — 11단계 한계(닫는 칸 ②).
- 우리 자료의 LNG 값(EPSIS 연료비단가)은 두 달쯤 늦게 따라오는 값이라, 3월 16일 상한 해제와 시장의 LNG 값 오름 가운데 무엇이 먼저였는지는 말할 수 없다(⑤ — 못 봄) — 11단계 한계(닫는 칸 ②).

</div>

## 3-가 — 9단계 인과지도에 물음 1 · 2에서 본 것을 더한 그림

**이 셀이 답하는 물음** — 9단계 그림에서 무엇을 더하고 무엇을 뺐나, 화살표마다 근거는 무엇인가.

**9단계 그림에서 바뀐 것**

| | 무엇 | 까닭 |
|---|---|---|
| 더한 변수 | 정부의 LNG 수요 관리(파란 글자) | 2-다 표 — 정부가 3월부터 LNG를 아끼려고 석탄 · 원전을 더 쓰겠다고 했다(확인 · 계획). 가격을 거치지 않고 석탄 · 원전에 닿는 길이다 |
| 더한 화살표 셋 | 호르무즈 봉쇄 → 수요 관리 · 수요 관리 → 석탄 발전 상한 · 수요 관리 → 원전 정비 | 아래 ② · ③ · ④ |
| 뺀 변수와 화살표 | 석탄 설비용량 → 석탄 발전 상한 | 석탄 발전소는 설비를 다 쓴 적이 없어서, 설비가 늘어도 석탄 발전량은 늘지 않는다 — 2023년 평일 평균으로 하루 중 가장 많이 돌린 19시에도 설비의 56.6%였다(4단계 3-다 「여력이 43% 남았다」). 2026년 차이에도 보이지 않았다 — 1월 석탄 발전량은 모델 예측과 거의 같았다(2-아 [표 1]). 9단계 4-나에서 「쓰이지 않으면 뺀다」로 남겨 둔 화살표다 |

그림은 변수 스물여섯 · 화살표 서른일곱입니다(9단계는 스물여섯 · 서른다섯).

**외부 자료** — 2-다 표와 9단계 4-가에서 고른 것을 다시 썼습니다(고르는 규칙은 그 셀 설명 칸). 새로 찾은 것은 신한울 2호기 상업운전 날짜 하나로, 원전 이름 줄을 확인으로 맞추려고 그림을 그린 뒤에 찾았습니다.

**화살표 ① ~ ⑧의 근거** — 그림 아래 줄과 같습니다. 「봤다」는 우리 자료로 본 것, 「확인 · 계획」은 외부 자료의 확인 정도, 「판단」은 시기만 맞는 것, 「2026년엔 아님」은 자료로 보니 2026년에는 그 화살표대로 움직이지 않은 것(⑧), 「못 봄」은 자료가 없어 확인하지 못한 것(⑤ · ⑦의 까닭)입니다. 화살표는 「보통 이렇게 이어진다」는 일반론이라, 「2026년엔 아님」이나 「못 봄」이어도 지우지 않습니다. 아래 문답은 이 둘을 「못 봄」 하나로 적었던 그림으로 했고, 그 그림은 문답 첫 칸에 있습니다.

| 화살표 | 근거 |
|---|---|
| ① 호르무즈 봉쇄 → LNG 가격 | LNG 값은 5월부터 평소 해와 달랐다(봤다 — 2-라 [표 1]) · 봉쇄 때문인지는 판단 |
| ② 호르무즈 봉쇄 → 정부의 LNG 수요 관리 | 정부 발표가 봉쇄를 까닭으로 들었다(확인 — 2-다 표 아래 「정부가 까닭으로 든 것」) |
| ③ 정부의 LNG 수요 관리 → 석탄 발전 상한 | 3월 16일 상한 해제(확인 — 2-다 표) · 해제 바로 뒤에는 석탄이 더 늘지 않았다(9단계 4-나) |
| ④ 정부의 LNG 수요 관리 → 원전 정비 | 원전 5기 5월까지 재가동(계획 — 2-다 표) · 원전은 1 ~ 6월 내내 모델 예측보다 적었다(2-나 [표 1]) |
| ⑤ LNG 가격 → 석탄 발전 상한 | 못 봄 · 우리 자료의 LNG 값(연료비단가)은 두 달쯤 늦게 따라와서(2-라 설명 칸), 3월 16일에 상한을 풀 때 시장의 LNG 값이 이미 올랐는지 볼 수 없다(아래 「못 봄 확인 (4)」) |
| ⑥ 원전 발전 → 석탄 발전 | 석탄과 LNG가 더 만든 양 가운데 석탄의 몫 1월 1% → 5월 91%(2-아 [표 2]) · 원전 대신인지는 판단 |
| ⑦ 전력 수요 → LNG 발전량 | 1 ~ 4월과 6월에 수요가 모델 예측보다 적었다(2-아 [표 1]) · 까닭은 못 봄 |
| ⑧ LNG 발전량 → LNG 가격 | 2026년엔 아님 · 2026년 LNG 발전량은 평소 범위였는데(9단계 물음 2) 값이 올랐다(2-라 [표 1]) |

그림 아래 줄에는 이 밖에 셋을 더 적었습니다 — 그 밖의 발전원(5 · 6월에 모델 예측보다 덜 만들었다, 대부분 신재생 · 기타였고, 그 가운데 태양광(역산 어림)보다 나머지가 컸다 · 나머지 가운데 무엇인지는 못 봄 — 2-바 · 2-사), 뺀 것(석탄 설비용량), 그리지 않은 것(LNG 가격 → LNG 발전량 — 값 싼 발전소부터 쓰는 시장이라 석탄이 LNG보다 싼 동안은 LNG 값이 올라도 쓰는 순서가 바뀌지 않아 7단계에서 지웠다. 석탄은 1 ~ 6월 내내 LNG보다 쌌다 — 2-가).

**그림을 읽을 때 같이 볼 것 둘** (그림에는 적지 않았습니다)

- **원전 설비용량** — 그림의 「원전 설비용량」은 전력거래소 연간 파일의 값이라 멈춘 원전까지 셉니다(9단계 4-다 [표 1] — 원전이 멈춘 날 앞뒤로 값이 그대로였다). 그래서 4단계의 「원자력 설비 2023 → 2025년 +1,342MW」를 실제로 발전할 수 있던 원전이 늘었다고 읽으면 안 됩니다. 그사이 새로 들어온 것은 신한울 2호기(1,400MW) 하나이고 멈춘 것은 고리 2 · 3 · 4호기와 한빛 1호기라(확인 — 9단계 4-가 설명 칸 「1월 1일에 이미 있던 일」의 기사들, 신한울 2호기는 [뉴스핌 2024-04-05](https://newspim.com/news/view/20240405001161)도), 실제로는 줄었다고 봅니다(판단). 2026년 1 ~ 6월 원전이 모델 예측보다 적었던 까닭 후보(9단계 결론 — 2026년 전에 멈춘 원전 넷)가 이것입니다.
- **석탄 발전 → 탄소 규제** — 석탄을 많이 때면 탄소 규제가 세진다는 화살표는 해 단위로 도는 정책이라, 1 ~ 6월을 읽는 데 쓸 곳이 없어 그리지 않았습니다(7단계 4-나 「긋지 않는 것」). 같이 넘어온 물량(재고)은 3-나 저량 · 유량 그림에서 그립니다.

**① 근거를 2-라 답변과 다르게 적은 까닭** — 2-라 답변과 물음 2를 닫는 답변에는 이 화살표 근거를 「봤다 — 두세 달 늦게」로 바꿀 수 있다고 적었습니다. 그림에는 「LNG 값은 5월부터 평소 해와 달랐다(봤다) · 봉쇄 때문인지는 판단」으로 적었습니다. LNG 값이 오른 것은 2-라 표에서 봤지만, 그것이 봉쇄 때문인지는 우리 자료로 가를 수 없기 때문입니다.

**그림 읽는 법** — 큰 굵은 글자는 모델이 보는 변수, 빨간 화살표는 7단계에서 2025년에 모델이 틀린 까닭으로 확인한 것, 화살표 위 ‖는 지연(효과가 늦게 나타난다), B · R은 고리(B는 균형, R은 강화)입니다. 파란 글자는 10단계에서 더한 변수이고, ① ~ ⑧은 10단계에서 근거를 단 화살표입니다(그 가운데 ② · ③ · ④는 새로 그은 화살표). 그림이 작게 보이면 그림 파일을 따로 여시면 됩니다 — `figures/인과지도_10단계.png`(창주 님이 Vensim에서 캡처) · `figures/인과지도_10단계.mdl`(Vensim 파일).

![인과지도 10단계](../figures/인과지도_10단계.png)

## 3-나 — LNG가 들어오고 쌓이고 쓰이는 양을 그린 저량 · 유량 그림

**이 셀이 답하는 물음** — LNG가 한국에 들어오고, 저장 탱크에 쌓이고, 발전과 도시가스에 쓰이는 길 가운데 이 프로젝트는 어디를 봤나.

| 넘어온 것 | 어디서 | 그림에서 |
|---|---|---|
| 「비축량에 따라 푸는 양(수요와 공급은 경제적으로 연결되어 있고, 가격에 따라 결정되니까?)」 · LNG가 모자라면 값과 관계없이 LNG 발전량을 줄여야 한다(클로드가 「가격 고리를 지운 까닭」에 적은 것) | 7단계 4-나 | ㉢ · 고리 B2 |
| 「lng 보관할 때 초저온으로 보관해야하는데 거기에 유지비용에 따라서 판단해서 lng수요가 줄수도 있나??」 | 7단계 4-나 | 그리지 않은 것 — 한국은 비축 의무와 겨울 대비 목표가 재고를 정한다(아래 문답 「보관 비용 확인」). 대신 ㉡ 재고 목표 |
| 「그래 이때는 당시니까 뭐 lng저장된 것도 있겠고..」 — 봉쇄 뒤 크게 벗어난 날이 눈에 띄지 않은 까닭으로 적으신 생각 | 9단계 3-나 「✍️ 내 답변」 | LNG 재고(네모) · 그림 아래 줄 「재고가 버틴 뒤 늦게 나타나는 영향」 |
| 짐작 ⑤의 물음 — 1단계 5번에 적으신 「…lng가 줄었다면 다른 곳에서 에너지원을 가져와야 했을 것인데 어디서 들고왔을까?」. 물음 2에서 「들여온 LNG는 이 프로젝트 자료에 없다」로 남겼다 | 1단계 물음 2 「✍️ 내가 정한 것」 5번 · 📌 물음 2 결론 「같이 정한 것」 2 | ㉠ |
| 정부의 LNG 수요 관리는 「재고를 지키려고 발전에 쓰는 LNG를 줄였다」는 이야기라 이 그림에 들어간다(클로드) | 2-나 아래 답변 | ㉤ · ㉥ |

**그림 읽는 법** — 저량 · 유량 그림은 쌓여 있는 양과 드나드는 양을 갈라 그리는 시스템 다이내믹스 그림입니다.

| 그림에서 | 뜻 | 이 그림의 것 |
|---|---|---|
| 네모 | 쌓이는 양(저량) — 어느 날 「지금 얼마나 있나」로 센다 | LNG 재고(저장 탱크에 있는 LNG) |
| 두 줄 화살표 | 드나드는 양(유량) — 「한 달에 얼마나 들어오고 나갔나」로 센다. 들어오는 쪽은 저량을 늘려 +, 나가는 쪽은 줄여 − | 들여온 LNG(+) · 발전에 쓴 LNG(−) · 도시가스로 쓴 LNG(−) |
| 구름 | 이 그림 밖 | 들여오기 전의 LNG · 쓰고 난 뒤 |
| 가는 화살표와 + · − | 3-가 인과지도와 같다 — + 는 같은 쪽으로, − 는 반대쪽으로 움직인다 | ㉠ ~ ㉦ |
| ‖ | 지연 — 효과가 늦게 나타난다 | ㉠ · ㉡ · ㉦(들여올 양을 정하거나 막힌 뒤 배가 들어오기까지 몇 주) |
| B1 · B2 | 균형 고리 — 저량을 지나 다시 그 저량으로 돌아온다(아래 문답 「고리 확인」) | B1 채우는 고리 · B2 아끼는 고리 |
| 초록 글자 | 이 프로젝트가 자료로 본 것 | LNG 발전량(1 ~ 6월 평소 범위 안 — 9단계) |

예 — 2-다 표의 「5월 말 한국 LNG 재고 약 42%」는 쌓이는 양이고, 「6월 한국 LNG 수입 326만 톤(예상)」은 드나드는 양입니다.

발전에 쓴 LNG는 LNG 발전량을 물음 1의 가정(발열량 · 효율)으로 톤으로 바꾼 양이라 초록이 아닙니다. 물음 1의 톤과 척도 발전에 쓴 LNG를 바꾼 양이고, 들여온 LNG의 양이나 실제로 들어온 배의 수가 아닙니다(닫는 칸 ② 물음 1 줄).

**고르는 규칙(외부 자료)** — 그림의 「들여온 LNG」와 「LNG 재고」로 들어오는 한국의 제도와 관행 가운데, 창주 님 7단계 물음(보관 비용이 들여오는 양을 줄이나)과 재고를 채우는 방식을 확인하는 것만 찾았습니다. 한국 자료를 먼저 썼고, 탱크에서 LNG가 증발하는 까닭처럼 나라와 관계없는 기술 자료는 해외 것도 썼습니다. 그림을 처음 그린 뒤 검토하면서 찾은 것이라, 결과를 보기 전에 정한 규칙이 아닙니다. 확인 정도는 2-다 표와 같고, 「기사 하나」는 일어난 일이지만 전한 기사가 하나뿐이라 확인으로 올리지 않은 것입니다 — 「추정」과 같이 「그런 자료가 있다」로만 씁니다.

**화살표 ㉠ ~ ㉦의 근거**

| 화살표 | 근거 |
|---|---|
| ㉠ 호르무즈 봉쇄 → 들여온 LNG (−, ‖) | 짐작 ⑤의 카타르 물량 · 못 봄 — 들여온 양은 우리 자료에 없다. 6월 수입이 2025년 6월보다 약 6% 적을 것이라는 추정이 있다(326만 톤 대 348만 톤 — 추정, 2-다 표) |
| ㉡ 재고 목표 → 들여온 LNG (+, ‖) | 가스공사의 재고 목표는 법정 비축 의무(하루 평균 판매량의 9일분 — 확인, [이투데이](https://www.etoday.co.kr/news/view/2020017) · [기후솔루션](https://forourclimate.org/ko/newsroom/1220))와 겨울 대비 목표(2022년에는 11월까지 저장설비의 90% — 계획, 기사 하나, [MTN 2022-08-08](https://news.mtn.co.kr/news-detail/2022080812285766652))가 정한다 · 2026년에 이 목표가 어땠는지는 못 봄 |
| ㉢ LNG 재고 → LNG 발전량 (+) | 재고가 모자라면 값과 관계없이 LNG 발전량을 줄인다 — 모자랄 때만 움직이는 화살표 · 못 봄 — 재고는 우리 자료에 없다. 5월 말 재고가 5년 평균 수준이었다는 추정이 있다(추정 — 2-다 표) |
| ㉣ LNG 발전량 → 발전에 쓴 LNG (+) | 환산 — 물음 1에서 9단계의 차이와 평소 범위를 가정(발열량 · 효율)으로 톤 · 배 몇 척으로 바꿨다 |
| ㉤ 호르무즈 봉쇄 → 정부의 LNG 수요 관리 (+) | 3-가 ②와 같다 — 정부 발표가 봉쇄를 까닭으로 들었다(확인 — 2-다 표 아래 「정부가 까닭으로 든 것」) |
| ㉥ 정부의 LNG 수요 관리 → LNG 발전량 (−) | 3-가 ③ · ④를 거치는 길을 한 화살표로 줄였다(석탄 상한 해제 3월 16일 — 확인 · 원전 재가동 5월까지 — 계획) · 판단 — 물음 2에서 석탄의 몫이 늘어난 것과 시기가 맞는 까닭 셋 가운데 하나 |
| ㉦ LNG 재고 → 들여온 LNG (−, ‖) | 재고가 목표보다 모자라면 더 들여오고 많으면 덜 들여온다. 들여오는 양의 70 ~ 80%가 장기 계약이라(기사 하나, [뉴시스 2026-06-05](https://www.newsis.com/view/NISX20260605_0003657637)) 바꿀 수 있는 것은 주로 현물 · 단기 계약이다(판단). 2022년에 가스공사가 재고가 낮아 현물 구매를 늘렸다(기사 하나 — MTN 2022-08-08) · 2026년은 못 봄 |

**고리**

| 고리 | 지나는 길과 부호 | 뜻 |
|---|---|---|
| B1 채우는 고리(균형) | LNG 재고 → 들여온 LNG(㉦, −) → LNG 재고(들어오는 두 줄 화살표, +) — 곱하면 − | 재고가 늘면 덜 들여와 다시 내려오고, 줄면 더 들여와 다시 올라가 재고가 목표 근처에 머문다 |
| B2 아끼는 고리(균형) | LNG 재고 → LNG 발전량(㉢, +) → 발전에 쓴 LNG(㉣, +) → LNG 재고(나가는 두 줄 화살표, −) — 곱하면 − | 재고가 모자라면 LNG 발전량을 줄여 재고가 덜 줄어든다 |

그림 아래 줄에는 이 밖에 넷을 더 적었습니다 — 도시가스로 쓴 LNG(못 봄 — 자료 없음), 재고가 버틴 뒤 늦게 나타나는 영향(비교 기간이 6월 30일까지라 그 뒤는 못 봄 — 9단계 3-다), 그리지 않은 것 둘(쌓아 두는 비용 → 들여온 LNG — 한국은 비축 의무와 겨울 대비 목표가 재고를 정해서 보관 비용이 들여오는 양을 정하지 않는다, 판단 · 아래 문답 「보관 비용 확인」 / LNG 가격 · 석탄 · 원전 — 3-가 인과지도에 있고, 이 그림은 물량 쪽만 그렸다).

그림 파일 — `figures/저량유량_10단계.png`(창주 님이 Vensim에서 캡처) · `figures/저량유량_10단계.mdl`(Vensim 파일). 아래 문답 가운데 앞 셋(재고 확인 · 경계 확인 · 보관 비용 확인)은 처음 그린 그림(`figures/저량유량_10단계_처음.png` · `.mdl`)을 보며 나눴고, 그 그림은 문답 첫 칸에 있습니다. 무엇을 왜 고쳤는지는 「보관 비용 확인」 아래 답변에 있습니다.

![저량 · 유량 그림 10단계](../figures/저량유량_10단계.png)